In [1]:
import pandas as pd

In [2]:
# Notebook 위치를 기준으로 상대경로를 사용해 원본 데이터를 불러옵니다.
df = pd.read_csv("../data/raw/Car details v3.csv")

In [3]:
# 전처리 전 데이터의 행과 열 개수를 확인합니다.
df.shape

(8128, 13)

In [4]:
# 전처리 전 완전히 동일한 중복 행의 개수를 확인합니다.
df.duplicated().sum()

np.int64(1202)

In [5]:
# 원본 df는 유지하고, 각 중복 그룹의 첫 번째 행만 남긴 별도 DataFrame을 만듭니다.
df_clean = df.drop_duplicates().copy()

In [6]:
# 중복 제거 후 데이터의 행과 열 개수를 확인합니다.
df_clean.shape

(6926, 13)

In [7]:
# 중복 제거로 실제 삭제된 행의 개수를 계산합니다.
len(df) - len(df_clean)

1202

In [8]:
# 중복 제거 후 완전히 동일한 중복 행이 남아 있는지 확인합니다.
df_clean.duplicated().sum()

np.int64(0)

In [9]:
# 중복 제거 후 각 컬럼의 결측치 개수를 확인합니다.
df_clean.isna().sum()

name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
mileage          208
engine           208
max_power        205
torque           209
seats            208
dtype: int64

In [10]:
# 결측치가 있는 5개 컬럼을 지정합니다.
missing_columns = [
    "mileage",
    "engine",
    "max_power",
    "torque",
    "seats",
]

# 각 행에서 결측인 컬럼 수와 그 분포를 확인합니다.
missing_count_per_row_clean = df_clean[missing_columns].isna().sum(axis=1)

missing_count_per_row_clean.value_counts().sort_index()

0    6717
1       1
4       3
5     205
Name: count, dtype: int64

In [11]:
# 5개 컬럼이 모두 동시에 결측인 행 수를 확인합니다.
df_clean[missing_columns].isna().all(axis=1).sum()

np.int64(205)

In [12]:
# 5개 중 정확히 4개 컬럼이 결측인 행을 모두 확인합니다.
df_clean.loc[missing_count_per_row_clean == 4]

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
575,Maruti Alto K10 LXI,2011,204999,97500,Petrol,Individual,Manual,First Owner,NaN,NaN,0,NaN,NaN
1442,Maruti Swift Dzire VDI Optional,2017,589000,41232,Diesel,Dealer,Manual,First Owner,NaN,NaN,0,NaN,NaN
2549,Tata Indica Vista Quadrajet LS,2012,240000,70000,Diesel,Individual,Manual,First Owner,NaN,NaN,0,NaN,NaN


In [13]:
# 5개 중 정확히 1개 컬럼만 결측인 행을 모두 확인합니다.
df_clean.loc[missing_count_per_row_clean == 1]

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
4933,Maruti Omni CNG,2000,80000,100000,CNG,Individual,Manual,Second Owner,10.9 km/kg,796 CC,bhp,NaN,8.0


In [14]:
# 5개 컬럼 중 하나라도 결측인 전체 행 수를 확인합니다.
df_clean[missing_columns].isna().any(axis=1).sum()

np.int64(209)

In [15]:
# mileage의 숫자 부분과 단위 표기를 확인합니다.
mileage_text = df_clean["mileage"].dropna().astype(str).str.strip()

mileage_parts = mileage_text.str.extract(
    r"^(?P<number>[0-9]+(?:\.[0-9]+)?)\s*(?P<unit>.*)$"
)

mileage_parts["unit"].value_counts(dropna=False)

unit
kmpl     6631
km/kg      87
Name: count, dtype: int64

In [16]:
# mileage의 숫자 부분 추출 실패 행 수를 확인합니다.
mileage_failure_mask = mileage_parts["number"].isna()
mileage_failure_count = mileage_failure_mask.sum()

mileage_failure_count

np.int64(0)

In [17]:
# mileage의 숫자 부분 추출 실패 행을 확인합니다.
mileage_failure_index = mileage_parts.index[mileage_failure_mask]
df_clean.loc[mileage_failure_index, ["name", "mileage"]]

,name,mileage


In [18]:
# engine의 숫자 부분과 단위 표기를 확인합니다.
engine_text = df_clean["engine"].dropna().astype(str).str.strip()

engine_parts = engine_text.str.extract(
    r"^(?P<number>[0-9]+(?:\.[0-9]+)?)\s*(?P<unit>.*)$"
)

engine_parts["unit"].value_counts(dropna=False)

unit
CC    6718
Name: count, dtype: int64

In [19]:
# engine의 숫자 부분 추출 실패 행 수를 확인합니다.
engine_failure_mask = engine_parts["number"].isna()
engine_failure_count = engine_failure_mask.sum()

engine_failure_count

np.int64(0)

In [20]:
# engine의 숫자 부분 추출 실패 행을 확인합니다.
engine_failure_index = engine_parts.index[engine_failure_mask]
df_clean.loc[engine_failure_index, ["name", "engine"]]

,name,engine


In [21]:
# max_power의 숫자 부분과 단위 표기를 확인합니다.
max_power_text = df_clean["max_power"].dropna().astype(str).str.strip()

max_power_parts = max_power_text.str.extract(
    r"^(?P<number>[0-9]+(?:\.[0-9]+)?)\s*(?P<unit>.*)$"
)

max_power_parts["unit"].value_counts(dropna=False)

unit
bhp    6717
          3
NaN       1
Name: count, dtype: int64

In [22]:
# max_power의 숫자 부분 추출 실패 행 수를 확인합니다.
max_power_failure_mask = max_power_parts["number"].isna()
max_power_failure_count = max_power_failure_mask.sum()

max_power_failure_count

np.int64(1)

In [23]:
# max_power의 숫자 부분 추출 실패 행을 모두 확인합니다.
max_power_failure_index = max_power_parts.index[max_power_failure_mask]
df_clean.loc[max_power_failure_index, ["name", "max_power"]]

,name,max_power
4933,Maruti Omni CNG,bhp


In [24]:
# 숫자는 추출되지만 max_power 단위가 빈 문자열인 행 수를 확인합니다.
max_power_empty_unit_mask = (
    max_power_parts["number"].notna()
    & max_power_parts["unit"].eq("")
)
max_power_empty_unit_count = max_power_empty_unit_mask.sum()

max_power_empty_unit_count

np.int64(3)

In [25]:
# 숫자는 추출되지만 max_power 단위가 빈 문자열인 행을 모두 확인합니다.
max_power_empty_unit_index = max_power_parts.index[max_power_empty_unit_mask]
df_clean.loc[max_power_empty_unit_index, ["name", "year", "max_power"]]

,name,year,max_power
575,Maruti Alto K10 LXI,2011,0
1442,Maruti Swift Dzire VDI Optional,2017,0
2549,Tata Indica Vista Quadrajet LS,2012,0


In [26]:
# max_power에서 추출한 숫자 중 0 이하인 행을 확인합니다.
max_power_number = pd.to_numeric(max_power_parts["number"], errors="coerce")
max_power_non_positive_mask = max_power_number.le(0)
max_power_non_positive_index = max_power_number.index[max_power_non_positive_mask]

df_clean.loc[max_power_non_positive_index, ["name", "year", "max_power"]]

,name,year,max_power
575,Maruti Alto K10 LXI,2011,0
1442,Maruti Swift Dzire VDI Optional,2017,0
2549,Tata Indica Vista Quadrajet LS,2012,0


In [27]:
# torque에서 사용되는 단위 유형을 확인합니다.
torque_text = df_clean["torque"].dropna().astype(str).str.strip()
torque_lower = torque_text.str.lower()

torque_nm_mask = torque_lower.str.contains("nm", regex=False)
torque_kgm_mask = torque_lower.str.contains("kgm", regex=False)
torque_other_mask = ~torque_nm_mask & ~torque_kgm_mask

pd.Series({
    "전체 non-null": len(torque_text),
    "nm 포함": torque_nm_mask.sum(),
    "kgm 포함": torque_kgm_mask.sum(),
    "nm과 kgm 미포함": torque_other_mask.sum(),
})

전체 non-null    6717
nm 포함          6227
kgm 포함          481
nm과 kgm 미포함      10
dtype: int64

In [28]:
# nm과 kgm 어느 것도 포함하지 않은 torque 고유값과 등장 횟수를 확인합니다.
torque_text[torque_other_mask].value_counts()

torque
210 / 1900           7
250@ 1250-5000rpm    1
510@ 1600-2400       1
110(11.2)@ 4800      1
Name: count, dtype: int64

In [29]:
# torque의 주요 표기 방식별 행 수를 확인합니다.
torque_at_sign_mask = torque_text.str.contains("@", regex=False)
torque_at_word_mask = torque_lower.str.contains("at", regex=False)
torque_range_mask = torque_text.str.contains("-", regex=False)

pd.Series({
    "@ 포함": torque_at_sign_mask.sum(),
    "at 포함": torque_at_word_mask.sum(),
    "- 포함": torque_range_mask.sum(),
})

@ 포함     6493
at 포함     212
- 포함     2196
dtype: int64

In [30]:
# @ 표기가 있는 torque 고유값 예시를 최대 10개 확인합니다.
torque_text[torque_at_sign_mask].drop_duplicates().head(10)

0            190Nm@ 2000rpm
1       250Nm@ 1500-2500rpm
2     12.7@ 2,700(kgm@ rpm)
4     11.5@ 4,500(kgm@ rpm)
5         113.75nm@ 4000rpm
6      7.8@ 4,500(kgm@ rpm)
7             59Nm@ 2500rpm
8       170Nm@ 1800-2400rpm
9            160Nm@ 2000rpm
10           248Nm@ 2250rpm
Name: torque, dtype: str

In [31]:
# at 표기가 있는 torque 고유값 예시를 최대 10개 확인합니다.
torque_text[torque_at_word_mask].drop_duplicates().head(10)

3      22.4 kgm at 1750-2750rpm
109           96 Nm at 3000 rpm
149          250 Nm at 2750 rpm
190           146Nm at 4800 rpm
193        14.9 KGM at 3000 RPM
226       11.4 kgm at 4,000 rpm
286      180 Nm at 1440-1500rpm
472         135 Nm at 2500  rpm
474     24 KGM at 1900-2750 RPM
641     260 Nm at 1800-2200 rpm
Name: torque, dtype: str

In [32]:
# 회전수 범위처럼 보이는 - 표기가 있는 torque 고유값 예시를 최대 10개 확인합니다.
torque_text[torque_range_mask].drop_duplicates().head(10)

1          250Nm@ 1500-2500rpm
3     22.4 kgm at 1750-2750rpm
8          170Nm@ 1800-2400rpm
15         115Nm@ 3500-3600rpm
19       219.7Nm@ 1500-2750rpm
39         320Nm@ 1700-2700rpm
41         250Nm@ 1750-2500rpm
47         343Nm@ 1400-3400rpm
48         200Nm@ 1400-3400rpm
49         200Nm@ 1250-4000rpm
Name: torque, dtype: str

In [33]:
# 조사 후에도 df_clean의 크기가 그대로인지 확인합니다.
df_clean.shape

(6926, 13)

In [34]:
# 조사 후에도 완전 중복 행이 없는지 확인합니다.
df_clean.duplicated().sum()

np.int64(0)

In [35]:
# nm과 kgm을 동시에 포함하는 torque 행 수를 확인합니다.
both_unit_mask = (
    torque_lower.str.contains("nm", na=False)
    & torque_lower.str.contains("kgm", na=False)
)

both_unit_mask.sum()

np.int64(1)

In [36]:
# nm과 kgm을 동시에 포함하는 원본 행을 확인합니다.
both_unit_index = torque_lower.index[both_unit_mask]
df_clean.loc[both_unit_index, ["name", "year", "torque"]]

,name,year,torque
778,Ford Endeavour Hurricane Limited Edition,2013,380Nm(38.7kgm)@ 2500rpm


In [37]:
# torque 단위 조건이 서로 겹치지 않도록 네 그룹의 행 수를 확인합니다.
nm_only_mask = (
    torque_lower.str.contains("nm", na=False)
    & ~torque_lower.str.contains("kgm", na=False)
)
kgm_only_mask = (
    ~torque_lower.str.contains("nm", na=False)
    & torque_lower.str.contains("kgm", na=False)
)
neither_unit_mask = (
    ~torque_lower.str.contains("nm", na=False)
    & ~torque_lower.str.contains("kgm", na=False)
)

exclusive_unit_counts = pd.Series({
    "nm만 포함": nm_only_mask.sum(),
    "kgm만 포함": kgm_only_mask.sum(),
    "nm과 kgm 모두 포함": both_unit_mask.sum(),
    "nm과 kgm 모두 미포함": neither_unit_mask.sum(),
})
exclusive_unit_counts.loc["네 그룹 합계"] = exclusive_unit_counts.sum()

exclusive_unit_counts

nm만 포함            6226
kgm만 포함            480
nm과 kgm 모두 포함        1
nm과 kgm 모두 미포함      10
네 그룹 합계           6717
dtype: int64

In [38]:
# 확인 후에도 df_clean의 크기가 그대로인지 확인합니다.
df_clean.shape

(6926, 13)

In [39]:
# 확인 후에도 완전 중복 행이 없는지 확인합니다.
df_clean.duplicated().sum()

np.int64(0)

In [40]:
# 기존 at 조건과 정확한 공백 포함 at 조건의 행 수를 비교합니다.
torque_exact_at_mask = torque_lower.str.contains(" at ", regex=False)

pd.Series({
    "기존 \"at\" 포함": torque_at_word_mask.sum(),
    "정확한 \" at \" 포함": torque_exact_at_mask.sum(),
})

기존 "at" 포함       212
정확한 " at " 포함    212
dtype: int64

In [41]:
# 두 at 조건의 결과가 다른 행 수를 확인합니다.
torque_at_difference_mask = torque_at_word_mask != torque_exact_at_mask

torque_at_difference_mask.sum()

np.int64(0)

In [42]:
# 두 at 조건에서 차이가 발생하는 torque 고유값과 등장 횟수를 확인합니다.
torque_text[torque_at_difference_mask].value_counts()

Series([], Name: count, dtype: int64)

In [43]:
# 확인 후에도 df_clean의 크기가 그대로인지 확인합니다.
df_clean.shape

(6926, 13)

In [44]:
# 확인 후에도 완전 중복 행이 없는지 확인합니다.
df_clean.duplicated().sum()

np.int64(0)

In [45]:
# mileage 숫자 부분을 확인용 숫자 Series로 변환합니다.
mileage_number = pd.to_numeric(
    mileage_parts["number"],
    errors="coerce",
)

mileage_unit = mileage_parts["unit"]

In [46]:
# 연료 유형과 mileage 단위의 관계를 확인합니다.
mileage_unit_check = pd.DataFrame({
    "fuel": df_clean.loc[mileage_unit.index, "fuel"],
    "mileage_unit": mileage_unit,
})

pd.crosstab(
    mileage_unit_check["fuel"],
    mileage_unit_check["mileage_unit"],
)

mileage_unit,km/kg,kmpl
fuel,,
CNG,52,0
Diesel,0,3658
LPG,35,0
Petrol,0,2973


In [47]:
# 각 연료 유형에서 나타나는 mileage 단위 종류 수를 확인합니다.
mileage_unit_check.groupby("fuel")["mileage_unit"].nunique()

fuel
CNG       1
Diesel    1
LPG       1
Petrol    1
Name: mileage_unit, dtype: int64

In [48]:
# 전체 non-null mileage 숫자값의 기본 분포를 확인합니다.
mileage_number.describe()

count    6718.00000
mean       19.46531
std         4.04915
min         0.00000
25%        16.80000
50%        19.44000
75%        22.50000
max        42.00000
Name: number, dtype: float64

In [49]:
# mileage 숫자값과 단위를 묶어 단위별 분포를 확인합니다.
mileage_numeric_check = pd.DataFrame({
    "mileage_number": mileage_number,
    "mileage_unit": mileage_unit,
})

mileage_numeric_check.groupby("mileage_unit")["mileage_number"].agg(
    ["count", "min", "median", "mean", "max"]
)

,count,min,median,mean,max
mileage_unit,,,,,
km/kg,87,10.9,21.94,21.810805,33.44
kmpl,6631,0.0,19.40,19.434536,42.00


In [50]:
# mileage 숫자값이 0 이하인 행 수를 확인합니다.
mileage_non_positive_mask = mileage_number.le(0)
mileage_non_positive_count = mileage_non_positive_mask.sum()

mileage_non_positive_count

np.int64(15)

In [51]:
# mileage 숫자값이 0 이하인 원본 행을 확인합니다.
mileage_non_positive_index = mileage_number.index[mileage_non_positive_mask]
df_clean.loc[mileage_non_positive_index, ["name", "year", "fuel", "mileage"]]

,name,year,fuel,mileage
644,Tata Indica Vista Aura Safire Anniversary Edition,2009,Petrol,0.0 kmpl
785,Hyundai Santro Xing GL,2009,Petrol,0.0 kmpl
1649,Hyundai Santro Xing GL,2008,Petrol,0.0 kmpl
1676,Mercedes-Benz M-Class ML 350 4Matic,2011,Diesel,0.0 kmpl
2137,Land Rover Freelander 2 TD4 HSE,2013,Diesel,0.0 kmpl
2366,Hyundai Santro Xing (Non-AC),2010,Petrol,0.0 kmpl
2725,Hyundai Santro Xing (Non-AC),2013,Petrol,0.0 kmpl
5276,Hyundai Santro Xing GL,2008,Petrol,0.0 kmpl
5843,Volkswagen Polo GT TSI BSIV,2014,Petrol,0.0 kmpl
5846,Volkswagen Polo GT TSI BSIV,2014,Petrol,0.0 kmpl


In [52]:
# non-null mileage 문자열 중 숫자 변환 실패 수를 확인합니다.
mileage_number.isna().sum()

np.int64(0)

In [53]:
# 조사 후에도 df_clean의 크기가 그대로인지 확인합니다.
df_clean.shape

(6926, 13)

In [54]:
# 조사 후에도 완전 중복 행이 없는지 확인합니다.
df_clean.duplicated().sum()

np.int64(0)

In [55]:
# mileage가 0 이하인 원본 행을 다시 지정합니다.
zero_mileage_index = mileage_number.index[mileage_number.le(0)]

zero_mileage_rows = df_clean.loc[
    zero_mileage_index,
    ["name", "year", "fuel", "mileage"],
]

zero_mileage_rows

,name,year,fuel,mileage
644,Tata Indica Vista Aura Safire Anniversary Edition,2009,Petrol,0.0 kmpl
785,Hyundai Santro Xing GL,2009,Petrol,0.0 kmpl
1649,Hyundai Santro Xing GL,2008,Petrol,0.0 kmpl
1676,Mercedes-Benz M-Class ML 350 4Matic,2011,Diesel,0.0 kmpl
2137,Land Rover Freelander 2 TD4 HSE,2013,Diesel,0.0 kmpl
2366,Hyundai Santro Xing (Non-AC),2010,Petrol,0.0 kmpl
2725,Hyundai Santro Xing (Non-AC),2013,Petrol,0.0 kmpl
5276,Hyundai Santro Xing GL,2008,Petrol,0.0 kmpl
5843,Volkswagen Polo GT TSI BSIV,2014,Petrol,0.0 kmpl
5846,Volkswagen Polo GT TSI BSIV,2014,Petrol,0.0 kmpl


In [56]:
# 0 mileage가 차종별로 몇 건인지 확인합니다.
zero_mileage_rows["name"].value_counts()

name
Hyundai Santro Xing GL                               5
Hyundai Santro Xing (Non-AC)                         2
Volkswagen Polo GT TSI BSIV                          2
Tata Indica Vista Aura Safire Anniversary Edition    1
Mercedes-Benz M-Class ML 350 4Matic                  1
Land Rover Freelander 2 TD4 HSE                      1
Mahindra Bolero Pik-Up FB 1.7T                       1
Mahindra Bolero Pik-Up CBC 1.7T                      1
Mercedes-Benz GLC 220d 4MATIC                        1
Name: count, dtype: int64

In [57]:
# mileage 숫자값이 0보다 큰 행만 비교용으로 준비합니다.
positive_mileage_index = mileage_number.index[mileage_number.gt(0)]

positive_mileage_rows = df_clean.loc[
    positive_mileage_index,
    ["name", "year", "fuel", "mileage"],
]

In [58]:
# 각 0 mileage 행과 같은 name의 양수 mileage를 확인합니다.
for idx, row in zero_mileage_rows.iterrows():
    same_name = positive_mileage_rows[
        positive_mileage_rows["name"].eq(row["name"])
    ]

    print("index: {}".format(idx))
    print("name: {}".format(row["name"]))
    print("year: {}".format(row["year"]))
    print("fuel: {}".format(row["fuel"]))
    print("현재 mileage: {}".format(row["mileage"]))
    print("같은 name의 양수 mileage:")
    print(same_name["mileage"].value_counts())
    print("-" * 50)

index: 644
name: Tata Indica Vista Aura Safire Anniversary Edition
year: 2009
fuel: Petrol
현재 mileage: 0.0 kmpl
같은 name의 양수 mileage:
Series([], Name: count, dtype: int64)
--------------------------------------------------
index: 785
name: Hyundai Santro Xing GL
year: 2009
fuel: Petrol
현재 mileage: 0.0 kmpl
같은 name의 양수 mileage:
Series([], Name: count, dtype: int64)
--------------------------------------------------
index: 1649
name: Hyundai Santro Xing GL
year: 2008
fuel: Petrol
현재 mileage: 0.0 kmpl
같은 name의 양수 mileage:
Series([], Name: count, dtype: int64)
--------------------------------------------------
index: 1676
name: Mercedes-Benz M-Class ML 350 4Matic
year: 2011
fuel: Diesel
현재 mileage: 0.0 kmpl
같은 name의 양수 mileage:
Series([], Name: count, dtype: int64)
--------------------------------------------------
index: 2137
name: Land Rover Freelander 2 TD4 HSE
year: 2013
fuel: Diesel
현재 mileage: 0.0 kmpl
같은 name의 양수 mileage:
Series([], Name: count, dtype: int64)
------------------------

In [59]:
# 각 0 mileage 행과 같은 name과 year의 양수 mileage를 확인합니다.
for idx, row in zero_mileage_rows.iterrows():
    same_name_year = positive_mileage_rows[
        positive_mileage_rows["name"].eq(row["name"])
        & positive_mileage_rows["year"].eq(row["year"])
    ]

    print("index: {}".format(idx))
    print("name: {}".format(row["name"]))
    print("year: {}".format(row["year"]))
    print("같은 name + year의 양수 mileage 기록 존재: {}".format(
        not same_name_year.empty
    ))
    print("같은 name + year의 양수 mileage:")
    print(same_name_year["mileage"].value_counts())
    print("-" * 50)

index: 644
name: Tata Indica Vista Aura Safire Anniversary Edition
year: 2009
같은 name + year의 양수 mileage 기록 존재: False
같은 name + year의 양수 mileage:
Series([], Name: count, dtype: int64)
--------------------------------------------------
index: 785
name: Hyundai Santro Xing GL
year: 2009
같은 name + year의 양수 mileage 기록 존재: False
같은 name + year의 양수 mileage:
Series([], Name: count, dtype: int64)
--------------------------------------------------
index: 1649
name: Hyundai Santro Xing GL
year: 2008
같은 name + year의 양수 mileage 기록 존재: False
같은 name + year의 양수 mileage:
Series([], Name: count, dtype: int64)
--------------------------------------------------
index: 1676
name: Mercedes-Benz M-Class ML 350 4Matic
year: 2011
같은 name + year의 양수 mileage 기록 존재: False
같은 name + year의 양수 mileage:
Series([], Name: count, dtype: int64)
--------------------------------------------------
index: 2137
name: Land Rover Freelander 2 TD4 HSE
year: 2013
같은 name + year의 양수 mileage 기록 존재: False
같은 name + year의 양수 mileag

In [60]:
# 0 mileage 행 중 비교 가능한 행 수를 집계합니다.
positive_names = set(positive_mileage_rows["name"])
positive_name_year_pairs = set(zip(
    positive_mileage_rows["name"],
    positive_mileage_rows["year"],
))

same_name_available = zero_mileage_rows["name"].isin(positive_names)
same_name_year_available = pd.Series(
    [
        (row["name"], row["year"]) in positive_name_year_pairs
        for _, row in zero_mileage_rows.iterrows()
    ],
    index=zero_mileage_rows.index,
)

pd.Series({
    "같은 name으로 비교 가능": same_name_available.sum(),
    "같은 name + year로 비교 가능": same_name_year_available.sum(),
    "같은 name에서도 비교 값 없음": (~same_name_available).sum(),
})

같은 name으로 비교 가능           0
같은 name + year로 비교 가능     0
같은 name에서도 비교 값 없음       15
dtype: int64

In [61]:
# 확인 후에도 df_clean의 크기가 그대로인지 확인합니다.
df_clean.shape

(6926, 13)

In [62]:
# 확인 후에도 완전 중복 행이 없는지 확인합니다.
df_clean.duplicated().sum()

np.int64(0)

In [63]:
# 실제 전처리를 적용할 별도 DataFrame을 만듭니다.
df_preprocessed = df_clean.copy()

In [64]:
# mileage에서 숫자 부분만 추출해 숫자형으로 변환합니다.
df_preprocessed["mileage"] = pd.to_numeric(
    df_preprocessed["mileage"].str.extract(
        r"^([0-9]+(?:\.[0-9]+)?)",
        expand=False,
    ),
    errors="coerce",
)

In [65]:
# 유효한 연비값으로 사용하지 않기로 한 0값을 결측값으로 변경합니다.
df_preprocessed.loc[
    df_preprocessed["mileage"].eq(0),
    "mileage",
] = pd.NA

In [66]:
# mileage가 숫자형으로 변환되었는지 확인합니다.
df_preprocessed["mileage"].dtype

dtype('float64')

In [67]:
# 전처리 후 mileage 결측치 수를 확인합니다.
df_preprocessed["mileage"].isna().sum()

np.int64(223)

In [68]:
# 전처리 후 mileage 0값이 남아 있는지 확인합니다.
df_preprocessed["mileage"].eq(0).sum()

np.int64(0)

In [69]:
# 0값을 결측 처리한 후 mileage 숫자값의 분포를 확인합니다.
df_preprocessed["mileage"].describe()

count    6703.000000
mean       19.508869
std         3.947453
min         9.000000
25%        16.800000
50%        19.490000
75%        22.540000
max        42.000000
Name: mileage, dtype: float64

In [70]:
# mileage 처리 전후 상태를 한 번에 비교합니다.
pd.Series({
    "처리 전 mileage 결측": df_clean["mileage"].isna().sum(),
    "처리 후 mileage 결측": df_preprocessed["mileage"].isna().sum(),
    "처리 후 mileage 0값": df_preprocessed["mileage"].eq(0).sum(),
})

처리 전 mileage 결측    208
처리 후 mileage 결측    223
처리 후 mileage 0값      0
dtype: int64

In [71]:
# mileage를 제외한 다른 컬럼이 변경되지 않았는지 확인합니다.
other_columns = df_clean.columns.difference(["mileage"])

df_clean[other_columns].equals(
    df_preprocessed[other_columns]
)

True

In [72]:
# mileage 전처리 후에도 전체 행과 열 수가 유지되는지 확인합니다.
df_preprocessed.shape

(6926, 13)

In [73]:
# mileage 전처리로 완전 중복 행이 생겼는지 확인합니다.
df_preprocessed.duplicated().sum()

np.int64(0)

In [74]:
# engine의 숫자 부분을 확인용 숫자 Series로 변환합니다.
engine_number = pd.to_numeric(
    df_preprocessed["engine"].str.extract(
        r"^([0-9]+(?:\.[0-9]+)?)",
        expand=False,
    ),
    errors="coerce",
)

In [75]:
# engine 숫자값의 기본 분포를 확인합니다.
engine_number.describe()

count    6718.000000
mean     1430.891337
std       493.493277
min       624.000000
25%      1197.000000
50%      1248.000000
75%      1498.000000
max      3604.000000
Name: engine, dtype: float64

In [76]:
# 원래 engine이 결측이 아닌 행 중 숫자 변환에 실패한 행을 확인합니다.
engine_non_null_mask = df_preprocessed["engine"].notna()

engine_conversion_failure_mask = (
    engine_non_null_mask
    & engine_number.isna()
)

engine_conversion_failure_mask.sum()

np.int64(0)

In [77]:
# engine 숫자 변환에 실패한 원본 행을 확인합니다.
df_preprocessed.loc[
    engine_conversion_failure_mask,
    ["name", "year", "engine"],
]

,name,year,engine


In [78]:
# engine 숫자값이 0 이하인 행 수를 확인합니다.
engine_number.le(0).sum()

np.int64(0)

In [79]:
# engine 숫자값이 0 이하인 원본 행을 확인합니다.
df_preprocessed.loc[
    engine_number.le(0),
    ["name", "year", "fuel", "engine"],
]

,name,year,fuel,engine


In [80]:
# 양수인 engine 고유값 중 가장 작은 10개를 확인합니다.
(
    engine_number[engine_number.gt(0)]
    .drop_duplicates()
    .sort_values()
    .head(10)
)

363     624.0
492     793.0
7       796.0
221     799.0
406     814.0
1327    909.0
503     936.0
11      993.0
986     995.0
36      998.0
Name: engine, dtype: float64

In [81]:
# 가장 작은 engine 값 5종에 해당하는 차량을 값별 최대 5행씩 확인합니다.
smallest_engine_values = (
    engine_number[engine_number.gt(0)]
    .drop_duplicates()
    .sort_values()
    .head(5)
)

smallest_engine_mask = engine_number.isin(smallest_engine_values)
smallest_engine_example_index = (
    engine_number[smallest_engine_mask]
    .groupby(engine_number[smallest_engine_mask])
    .head(5)
    .sort_values()
    .index
)

df_preprocessed.loc[
    smallest_engine_example_index,
    ["name", "year", "fuel", "engine"],
]

,name,year,fuel,engine
363,Tata Nano STD,2012,Petrol,624 CC
1267,Tata Nano Cx BSIV,2010,Petrol,624 CC
1217,Tata Nano CX,2013,Petrol,624 CC
709,Tata Nano Cx,2011,Petrol,624 CC
1425,Tata Nano XTA,2015,Petrol,624 CC
492,Maruti Celerio LDi,2016,Diesel,793 CC
1333,Maruti Celerio VDi,2015,Diesel,793 CC
2035,Maruti Celerio ZDi,2015,Diesel,793 CC
7372,Maruti Celerio VDi,2016,Diesel,793 CC
5016,Maruti Celerio ZDi,2015,Diesel,793 CC


In [82]:
# engine의 기존 결측치 수를 다시 확인합니다.
df_preprocessed["engine"].isna().sum()

np.int64(208)

In [83]:
# engine 조사 후에도 전체 행과 열 수가 유지되는지 확인합니다.
df_preprocessed.shape

(6926, 13)

In [84]:
# engine 조사 후에도 완전 중복 행이 없는지 확인합니다.
df_preprocessed.duplicated().sum()

np.int64(0)

In [85]:
# engine에서 숫자 부분만 추출해 숫자형으로 변환합니다.
df_preprocessed["engine"] = pd.to_numeric(
    df_preprocessed["engine"].str.extract(
        r"^([0-9]+(?:\.[0-9]+)?)",
        expand=False,
    ),
    errors="coerce",
)

In [86]:
# engine이 숫자형으로 변환되었는지 확인합니다.
df_preprocessed["engine"].dtype

dtype('float64')

In [87]:
# 숫자 변환 후 engine 결측치 수를 확인합니다.
df_preprocessed["engine"].isna().sum()

np.int64(208)

In [88]:
# 숫자 변환 후 engine 값의 분포를 확인합니다.
df_preprocessed["engine"].describe()

count    6718.000000
mean     1430.891337
std       493.493277
min       624.000000
25%      1197.000000
50%      1248.000000
75%      1498.000000
max      3604.000000
Name: engine, dtype: float64

In [89]:
# 숫자 변환 후 engine에 0 이하 값이 있는지 확인합니다.
df_preprocessed["engine"].le(0).sum()

np.int64(0)

In [90]:
# 문자열 단위가 제거된 engine 값의 예시를 확인합니다.
df_preprocessed.loc[
    df_preprocessed["engine"].notna(),
    ["name", "engine"],
].head(10)

,name,engine
0,Maruti Swift Dzire VDI,1248.0
1,Skoda Rapid 1.5 TDI Ambition,1498.0
2,Honda City 2017-2020 EXi,1497.0
3,Hyundai i20 Sportz Diesel,1396.0
4,Maruti Swift VXI BSIII,1298.0
5,Hyundai Xcent 1.2 VTVT E Plus,1197.0
6,Maruti Wagon R LXI DUO BSIII,1061.0
7,Maruti 800 DX BSII,796.0
8,Toyota Etios VXD,1364.0
9,Ford Figo Diesel Celebration Edition,1399.0


In [91]:
# engine 전처리 후에도 mileage 상태가 유지되는지 확인합니다.
pd.Series({
    "mileage 결측": df_preprocessed["mileage"].isna().sum(),
    "mileage 0값": df_preprocessed["mileage"].eq(0).sum(),
})

mileage 결측    223
mileage 0값      0
dtype: int64

In [92]:
# engine 전처리 후에도 전체 행과 열 수가 유지되는지 확인합니다.
df_preprocessed.shape

(6926, 13)

In [93]:
# engine 전처리 후 완전 중복 행 수를 확인합니다.
df_preprocessed.duplicated().sum()

np.int64(0)

In [94]:
# max_power의 숫자 부분을 확인용 숫자 Series로 변환합니다.
max_power_number = pd.to_numeric(
    df_preprocessed["max_power"].str.extract(
        r"^([0-9]+(?:\.[0-9]+)?)",
        expand=False,
    ),
    errors="coerce",
)

In [95]:
# max_power 숫자값의 분포를 확인합니다.
max_power_number.describe()

count    6720.000000
mean       87.726919
std        31.771619
min         0.000000
25%        67.100000
50%        81.830000
75%       100.000000
max       400.000000
Name: max_power, dtype: float64

In [96]:
# 원래 값이 존재하지만 숫자로 변환되지 않은 행을 구분합니다.
max_power_non_null_mask = df_preprocessed["max_power"].notna()

max_power_conversion_failure_mask = (
    max_power_non_null_mask
    & max_power_number.isna()
)

max_power_conversion_failure_mask.sum()

np.int64(1)

In [97]:
# 숫자 변환에 실패한 원본 행을 확인합니다.
df_preprocessed.loc[
    max_power_conversion_failure_mask,
    ["name", "year", "fuel", "engine", "max_power"],
]

,name,year,fuel,engine,max_power
4933,Maruti Omni CNG,2000,CNG,796.0,bhp


In [98]:
# max_power 숫자값 중 0 이하인 행 수를 확인합니다.
max_power_number.le(0).sum()

np.int64(3)

In [99]:
# max_power 숫자값이 0 이하인 원본 행을 확인합니다.
zero_max_power_rows = df_preprocessed.loc[
    max_power_number.le(0),
    ["name", "year", "fuel", "engine", "max_power"],
]

zero_max_power_rows

,name,year,fuel,engine,max_power
575,Maruti Alto K10 LXI,2011,Petrol,NaN,0
1442,Maruti Swift Dzire VDI Optional,2017,Diesel,NaN,0
2549,Tata Indica Vista Quadrajet LS,2012,Diesel,NaN,0


In [100]:
# 0값 차량과 동일한 name의 양수 max_power 기록을 확인합니다.
positive_max_power_mask = max_power_number.gt(0)

for idx, row in zero_max_power_rows.iterrows():
    same_name_mask = (
        df_preprocessed["name"].eq(row["name"])
        & positive_max_power_mask
    )
    positive_values = sorted(
        max_power_number.loc[same_name_mask].unique().tolist()
    )

    print(f"index: {idx}")
    print(f"name: {row['name']}")
    print(f"year: {row['year']}")
    print(f"원본 max_power: {row['max_power']}")
    print(f"동일 name의 양수 max_power 존재: {bool(positive_values)}")
    print(f"양수 max_power 고유값: {positive_values}")
    print("-" * 50)

index: 575
name: Maruti Alto K10 LXI
year: 2011
원본 max_power: 0
동일 name의 양수 max_power 존재: True
양수 max_power 고유값: [67.05, 67.1]
--------------------------------------------------
index: 1442
name: Maruti Swift Dzire VDI Optional
year: 2017
원본 max_power: 0
동일 name의 양수 max_power 존재: True
양수 max_power 고유값: [74.0]
--------------------------------------------------
index: 2549
name: Tata Indica Vista Quadrajet LS
year: 2012
원본 max_power: 0
동일 name의 양수 max_power 존재: True
양수 max_power 고유값: [74.0]
--------------------------------------------------


In [101]:
# 0값 차량과 동일한 name과 year의 양수 max_power 기록을 확인합니다.
for idx, row in zero_max_power_rows.iterrows():
    same_name_year_mask = (
        df_preprocessed["name"].eq(row["name"])
        & df_preprocessed["year"].eq(row["year"])
        & positive_max_power_mask
    )
    positive_values = sorted(
        max_power_number.loc[same_name_year_mask].unique().tolist()
    )

    print(f"index: {idx}")
    print(f"name: {row['name']}")
    print(f"year: {row['year']}")
    print(f"동일 name + year의 양수 max_power 존재: {bool(positive_values)}")
    print(f"양수 max_power 고유값: {positive_values}")
    print("-" * 50)

index: 575
name: Maruti Alto K10 LXI
year: 2011
동일 name + year의 양수 max_power 존재: True
양수 max_power 고유값: [67.1]
--------------------------------------------------
index: 1442
name: Maruti Swift Dzire VDI Optional
year: 2017
동일 name + year의 양수 max_power 존재: True
양수 max_power 고유값: [74.0]
--------------------------------------------------
index: 2549
name: Tata Indica Vista Quadrajet LS
year: 2012
동일 name + year의 양수 max_power 존재: True
양수 max_power 고유값: [74.0]
--------------------------------------------------


In [102]:
# 숫자 변환에 실패한 행과 동일 차량의 숫자형 max_power 기록을 확인합니다.
failed_max_power_rows = df_preprocessed.loc[
    max_power_conversion_failure_mask,
    ["name", "year", "max_power"],
]

for idx, row in failed_max_power_rows.iterrows():
    same_name_mask = (
        df_preprocessed["name"].eq(row["name"])
        & positive_max_power_mask
    )
    same_name_year_mask = (
        same_name_mask
        & df_preprocessed["year"].eq(row["year"])
    )
    same_name_values = sorted(
        max_power_number.loc[same_name_mask].unique().tolist()
    )
    same_name_year_values = sorted(
        max_power_number.loc[same_name_year_mask].unique().tolist()
    )

    print(f"index: {idx}")
    print(f"name: {row['name']}")
    print(f"year: {row['year']}")
    print(f"원본 max_power: {row['max_power']}")
    print(f"동일 name의 양수 max_power 존재: {bool(same_name_values)}")
    print(f"동일 name의 양수 max_power 고유값: {same_name_values}")
    print(f"동일 name + year의 양수 max_power 존재: {bool(same_name_year_values)}")
    print(f"동일 name + year의 양수 max_power 고유값: {same_name_year_values}")

index: 4933
name: Maruti Omni CNG
year: 2000
원본 max_power:  bhp
동일 name의 양수 max_power 존재: False
동일 name의 양수 max_power 고유값: []
동일 name + year의 양수 max_power 존재: False
동일 name + year의 양수 max_power 고유값: []


In [103]:
# 가장 작은 양수 max_power 고유값 10개를 확인합니다.
smallest_positive_max_power_values = (
    max_power_number[max_power_number.gt(0)]
    .drop_duplicates()
    .sort_values()
    .head(10)
)

smallest_positive_max_power_values

3466    32.80
29      34.20
32      35.00
3378    35.50
7       37.00
1217    37.48
363     37.50
3942    38.00
7300    38.40
35      40.30
Name: max_power, dtype: float64

In [104]:
# 가장 작은 max_power 값 5종에 해당하는 차량을 값별 최대 5행 확인합니다.
smallest_five_max_power_values = smallest_positive_max_power_values.head(5)

small_max_power_examples = df_preprocessed.loc[
    max_power_number.isin(smallest_five_max_power_values),
    ["name", "year", "fuel", "engine", "max_power"],
].copy()
small_max_power_examples["max_power_number"] = max_power_number.loc[
    small_max_power_examples.index
]

(
    small_max_power_examples
    .sort_values(["max_power_number", "name", "year"])
    .groupby("max_power_number", group_keys=False)
    .head(5)
    [["name", "year", "fuel", "engine", "max_power"]]
)

,name,year,fuel,engine,max_power
6007,Maruti Omni LPG CARGO BSIII W IMMOBILISER,2007,LPG,796.0,32.8 bhp
3466,Maruti Omni LPG CARGO BSIII W IMMOBILISER,2010,LPG,796.0,32.8 bhp
2453,Maruti Omni E 8 Str STD,2016,Petrol,796.0,34.2 bhp
2454,Maruti Omni E 8 Str STD,2016,Petrol,796.0,34.2 bhp
5243,Maruti Omni E MPI STD BS IV,2014,Petrol,796.0,34.2 bhp
289,Maruti Omni E MPI STD BS IV,2016,Petrol,796.0,34.2 bhp
814,Maruti Omni E MPI STD BS IV,2016,Petrol,796.0,34.2 bhp
5978,Maruti Omni 5 Seater BSIV,2011,Petrol,796.0,35 bhp
2327,Maruti Omni 5 Str STD,1998,Petrol,796.0,35 bhp
3515,Maruti Omni 8 Seater BSII,2008,Petrol,796.0,35 bhp


In [105]:
# max_power 원본 컬럼의 기존 결측치 수를 다시 확인합니다.
df_preprocessed["max_power"].isna().sum()

np.int64(205)

In [106]:
# max_power 조사 후에도 전체 행과 열 수가 유지되는지 확인합니다.
df_preprocessed.shape

(6926, 13)

In [107]:
# max_power 조사 후에도 완전 중복 행이 없는지 확인합니다.
df_preprocessed.duplicated().sum()

np.int64(0)

In [108]:
# max_power에서 숫자 부분만 추출해 숫자형으로 변환합니다.
df_preprocessed["max_power"] = pd.to_numeric(
    df_preprocessed["max_power"].str.extract(
        r"^([0-9]+(?:\.[0-9]+)?)",
        expand=False,
    ),
    errors="coerce",
)

In [109]:
# 유효한 최고출력으로 보기 어려운 max_power 0값을 결측값으로 처리합니다.
df_preprocessed.loc[
    df_preprocessed["max_power"].eq(0),
    "max_power",
] = pd.NA

In [110]:
# max_power가 숫자형으로 변환되었는지 확인합니다.
df_preprocessed["max_power"].dtype

dtype('float64')

In [111]:
# 숫자 변환과 0값 처리 후 max_power 결측치 수를 확인합니다.
df_preprocessed["max_power"].isna().sum()

np.int64(209)

In [112]:
# 처리 후 max_power에 0값이 남아 있는지 확인합니다.
df_preprocessed["max_power"].eq(0).sum()

np.int64(0)

In [113]:
# 처리 후 max_power 숫자값의 분포를 확인합니다.
df_preprocessed["max_power"].describe()

count    6717.000000
mean       87.766100
std        31.724555
min        32.800000
25%        67.100000
50%        81.830000
75%       100.000000
max       400.000000
Name: max_power, dtype: float64

In [114]:
# 조사 단계에서 확인한 예외행 4개의 max_power 상태를 확인합니다.
df_preprocessed.loc[
    [575, 1442, 2549, 4933],
    ["name", "year", "max_power"],
]

,name,year,max_power
575,Maruti Alto K10 LXI,2011,NaN
1442,Maruti Swift Dzire VDI Optional,2017,NaN
2549,Tata Indica Vista Quadrajet LS,2012,NaN
4933,Maruti Omni CNG,2000,NaN


In [115]:
# max_power 전처리 후에도 mileage와 engine 상태가 유지되는지 확인합니다.
pd.Series({
    "mileage 결측": df_preprocessed["mileage"].isna().sum(),
    "mileage 0값": df_preprocessed["mileage"].eq(0).sum(),
    "engine 결측": df_preprocessed["engine"].isna().sum(),
    "engine 0 이하": df_preprocessed["engine"].le(0).sum(),
})

mileage 결측     223
mileage 0값       0
engine 결측      208
engine 0 이하      0
dtype: int64

In [116]:
# max_power 전처리 후에도 전체 행과 열 수가 유지되는지 확인합니다.
df_preprocessed.shape

(6926, 13)

In [117]:
# max_power 전처리 후 완전 중복 행 수를 확인합니다.
df_preprocessed.duplicated().sum()

np.int64(0)

In [118]:
# torque의 결측치와 non-null 행 수를 다시 확인합니다.
pd.Series({
    "결측": df_preprocessed["torque"].isna().sum(),
    "non-null": df_preprocessed["torque"].notna().sum(),
})

결측           209
non-null    6717
dtype: int64

In [119]:
# 실제 컬럼을 변경하지 않고 torque 문자열을 조사용으로 정리합니다.
torque_text = df_preprocessed["torque"].str.strip()
torque_lower = torque_text.str.lower()

In [120]:
# torque 단위 유형을 서로 겹치지 않는 네 그룹으로 분류합니다.
torque_nm_mask = torque_lower.str.contains("nm", regex=False, na=False)
torque_kgm_mask = torque_lower.str.contains("kgm", regex=False, na=False)

nm_only_mask = torque_nm_mask & ~torque_kgm_mask
kgm_only_mask = torque_kgm_mask & ~torque_nm_mask
both_unit_mask = torque_nm_mask & torque_kgm_mask
neither_unit_mask = ~torque_nm_mask & ~torque_kgm_mask & torque_text.notna()

pd.Series({
    "Nm only": nm_only_mask.sum(),
    "kgm only": kgm_only_mask.sum(),
    "both": both_unit_mask.sum(),
    "neither": neither_unit_mask.sum(),
    "합계": (
        nm_only_mask.sum()
        + kgm_only_mask.sum()
        + both_unit_mask.sum()
        + neither_unit_mask.sum()
    ),
})

Nm only     6226
kgm only     480
both           1
neither       10
합계          6717
dtype: int64

In [121]:
# Nm과 kgm을 모두 포함한 행을 확인합니다.
df_preprocessed.loc[
    both_unit_mask,
    ["name", "year", "torque"],
]

,name,year,torque
778,Ford Endeavour Hurricane Limited Edition,2013,380Nm(38.7kgm)@ 2500rpm


In [122]:
# Nm과 kgm 어느 단위도 명시되지 않은 10행을 모두 확인합니다.
neither_unit_rows = df_preprocessed.loc[
    neither_unit_mask,
    ["name", "year", "fuel", "engine", "max_power", "torque"],
]

neither_unit_rows

,name,year,fuel,engine,max_power,torque
140,Skoda Superb LK 1.8 TSI AT,2018,Petrol,1798.0,177.46,250@ 1250-5000rpm
1676,Mercedes-Benz M-Class ML 350 4Matic,2011,Diesel,2987.0,165.00,510@ 1600-2400
2000,Honda Jazz Select Edition Active,2011,Petrol,1198.0,90.00,110(11.2)@ 4800
4373,Skoda Octavia Classic 1.9 TDI MT,2006,Diesel,1896.0,66.00,210 / 1900
5572,Skoda Octavia Classic 1.9 TDI MT,2007,Diesel,1896.0,66.00,210 / 1900
5785,Skoda Octavia Ambiente 1.9 TDI MT,2007,Diesel,1896.0,66.00,210 / 1900
6418,Skoda Octavia Ambiente 1.9 TDI MT,2003,Diesel,1896.0,66.00,210 / 1900
7154,Skoda Octavia Rider 1.9 AT TDI,2008,Diesel,1896.0,66.00,210 / 1900
7296,Skoda Octavia Ambiente 1.9 TDI,2010,Diesel,1896.0,66.00,210 / 1900
7532,Skoda Octavia Ambiente 1.9 TDI MT,2007,Diesel,1896.0,66.00,210 / 1900


In [123]:
# 단위가 명시되지 않은 torque 문자열별 행 수를 확인합니다.
torque_text[neither_unit_mask].value_counts()

torque
210 / 1900           7
250@ 1250-5000rpm    1
510@ 1600-2400       1
110(11.2)@ 4800      1
Name: count, dtype: int64

In [124]:
# torque 구분자 유형별 포함 행 수를 확인합니다.
torque_at_sign_mask = torque_text.str.contains("@", regex=False, na=False)
torque_at_word_mask = torque_lower.str.contains(" at ", regex=False, na=False)
torque_slash_mask = torque_text.str.contains("/", regex=False, na=False)

pd.Series({
    "@ 포함": torque_at_sign_mask.sum(),
    '" at " 포함': torque_at_word_mask.sum(),
    "/ 포함": torque_slash_mask.sum(),
})

@ 포함         6493
" at " 포함     212
/ 포함           22
dtype: int64

In [125]:
# torque 문자열에서 첫 번째 숫자를 조사용으로 추출합니다.
torque_first_number = pd.to_numeric(
    torque_text.str.extract(
        r"([0-9]+(?:\.[0-9]+)?)",
        expand=False,
    ),
    errors="coerce",
)

In [126]:
# non-null torque 중 첫 번째 숫자 추출에 실패한 행 수를 확인합니다.
(
    torque_text.notna()
    & torque_first_number.isna()
).sum()

np.int64(0)

In [127]:
# torque 첫 번째 숫자의 전체 분포를 확인합니다.
torque_first_number.describe()

count    6717.000000
mean      160.854853
std        91.630280
min         4.800000
25%        96.000000
50%       146.000000
75%       200.000000
max       789.000000
Name: torque, dtype: float64

In [128]:
# kgm만 포함된 torque의 첫 번째 숫자 분포를 확인합니다.
torque_first_number[kgm_only_mask].describe()

count    480.000000
mean      21.975417
std       25.410323
min        4.800000
25%       12.500000
50%       16.300000
75%       22.400000
max      190.000000
Name: torque, dtype: float64

In [129]:
# kgm만 포함된 torque의 가장 작은 고유값 10개를 확인합니다.
pd.Series({
    "가장 작은 고유값": sorted(
        torque_first_number[kgm_only_mask].dropna().unique()
    )[:10],
    "가장 큰 고유값": sorted(
        torque_first_number[kgm_only_mask].dropna().unique(),
        reverse=True,
    )[:10],
})

가장 작은 고유값    [4.8, 5.7, 6.0, 6.1, 7.8, 8.5, 8.6, 9.2, 9.8, ...
가장 큰 고유값     [190.0, 145.0, 130.0, 115.0, 110.0, 53.0, 51.0...
dtype: object

In [130]:
# Nm만 포함된 torque의 첫 번째 숫자 분포를 확인합니다.
torque_first_number[nm_only_mask].describe()

count    6226.000000
mean      171.409227
std        85.897174
min        48.000000
25%       110.000000
50%       160.000000
75%       200.000000
max       789.000000
Name: torque, dtype: float64

In [131]:
# Nm만 포함된 torque의 가장 작은 고유값과 가장 큰 고유값을 확인합니다.
pd.Series({
    "가장 작은 고유값": sorted(
        torque_first_number[nm_only_mask].dropna().unique()
    )[:10],
    "가장 큰 고유값": sorted(
        torque_first_number[nm_only_mask].dropna().unique(),
        reverse=True,
    )[:10],
})

가장 작은 고유값    [48.0, 51.0, 57.0, 59.0, 60.0, 62.0, 69.0, 71....
가장 큰 고유값     [789.0, 640.0, 620.0, 619.0, 600.0, 580.0, 560...
dtype: object

In [132]:
# 괄호가 포함된 torque 행 수와 고유 문자열 수를 확인합니다.
torque_parenthesis_mask = torque_text.str.contains(
    r"[()]",
    regex=True,
    na=False,
)
parenthesis_value_counts = torque_text[torque_parenthesis_mask].value_counts()

pd.Series({
    "괄호 포함 행 수": torque_parenthesis_mask.sum(),
    "괄호 포함 고유 문자열 수": parenthesis_value_counts.size,
})

괄호 포함 행 수         379
괄호 포함 고유 문자열 수     69
dtype: int64

In [133]:
# 괄호가 포함된 torque 고유 문자열을 빈도순으로 최대 30개 확인합니다.
parenthesis_value_counts.head(30)

torque
20.4@ 1400-3400(kgm@ rpm)      77
12.7@ 2,700(kgm@ rpm)          26
16.3@ 2,000(kgm@ rpm)          25
24@ 1,900-2,750(kgm@ rpm)      24
13.5@ 2,500(kgm@ rpm)          22
12.5@ 3,500(kgm@ rpm)          12
8.5@ 2,500(kgm@ rpm)           12
16@ 2,000(kgm@ rpm)            10
12.5@ 2,500(kgm@ rpm)          10
145@ 4,100(kgm@ rpm)           10
11.5@ 4,500(kgm@ rpm)           9
9.8@ 3,000(kgm@ rpm)            9
8.6@ 3,500(kgm@ rpm)            9
14.9@ 3,000(kgm@ rpm)           8
19@ 1,800(kgm@ rpm)             8
16.1@ 4,200(kgm@ rpm)           7
7.8@ 4,500(kgm@ rpm)            6
28.3@ 1,700-2,200(kgm@ rpm)     6
12.5@ 3,000(kgm@ rpm)           5
115@ 2,500(kgm@ rpm)            4
130@ 2500(kgm@ rpm)             4
25.5@ 1,900(kgm@ rpm)           4
14.3@ 1,800-3,000(kgm@ rpm)     3
13@ 2,500(kgm@ rpm)             3
12@ 3,500(kgm@ rpm)             3
10.7@ 2,500(kgm@ rpm)           3
11.2@ 4,400(kgm@ rpm)           3
13.5@ 4,800(kgm@ rpm)           3
21.4@ 1,900(kgm@ rpm)           3
48@ 3,0

In [134]:
# torque 문자열별 숫자 개수를 계산하고 분포를 확인합니다.
torque_number_count = torque_text.str.findall(
    r"[0-9]+(?:\.[0-9]+)?"
).str.len()

torque_number_count.value_counts().sort_index()

torque
1.0       4
2.0    4237
3.0    2425
4.0       2
5.0      49
Name: count, dtype: int64

In [135]:
# rpm 표현과 관련된 기호 및 문자열 포함 행 수를 확인합니다.
torque_hyphen_mask = torque_text.str.contains("-", regex=False, na=False)
torque_comma_mask = torque_text.str.contains(",", regex=False, na=False)
torque_rpm_mask = torque_lower.str.contains("rpm", regex=False, na=False)

pd.Series({
    "- 포함": torque_hyphen_mask.sum(),
    ", 포함": torque_comma_mask.sum(),
    "rpm 포함": torque_rpm_mask.sum(),
    "숫자 1개": torque_number_count.eq(1).sum(),
    "숫자 2개": torque_number_count.eq(2).sum(),
    "숫자 3개 이상": torque_number_count.ge(3).sum(),
})

- 포함        2196
, 포함         327
rpm 포함      6698
숫자 1개          4
숫자 2개       4237
숫자 3개 이상    2476
dtype: int64

In [136]:
# 대표 형식별 torque 고유 문자열을 최대 10개씩 확인합니다.
torque_format_masks = {
    "Nm + 단일 rpm": nm_only_mask & torque_rpm_mask & ~torque_hyphen_mask,
    "Nm + rpm 범위": nm_only_mask & torque_rpm_mask & torque_hyphen_mask,
    "kgm + 단일 rpm": kgm_only_mask & torque_rpm_mask & ~torque_hyphen_mask,
    "kgm + rpm 범위": kgm_only_mask & torque_rpm_mask & torque_hyphen_mask,
    "Nm과 kgm 둘 다 포함": both_unit_mask,
    "단위 없음": neither_unit_mask,
    "괄호 포함": torque_parenthesis_mask,
}

for label, mask in torque_format_masks.items():
    examples = torque_text[mask].drop_duplicates().head(10).tolist()
    print(label)
    for example in examples:
        print(f"- {example}")
    print("-" * 50)

Nm + 단일 rpm
- 190Nm@ 2000rpm
- 113.75nm@ 4000rpm
- 59Nm@ 2500rpm
- 160Nm@ 2000rpm
- 248Nm@ 2250rpm
- 78Nm@ 4500rpm
- 84Nm@ 3500rpm
- 200Nm@ 1750rpm
- 62Nm@ 3000rpm
- 114Nm@ 3500rpm
--------------------------------------------------
Nm + rpm 범위
- 250Nm@ 1500-2500rpm
- 170Nm@ 1800-2400rpm
- 115Nm@ 3500-3600rpm
- 219.7Nm@ 1500-2750rpm
- 320Nm@ 1700-2700rpm
- 250Nm@ 1750-2500rpm
- 343Nm@ 1400-3400rpm
- 200Nm@ 1400-3400rpm
- 200Nm@ 1250-4000rpm
- 400Nm@ 2000-2500rpm
--------------------------------------------------
kgm + 단일 rpm
- 12.7@ 2,700(kgm@ rpm)
- 11.5@ 4,500(kgm@ rpm)
- 7.8@ 4,500(kgm@ rpm)
- 6.1kgm@ 3000rpm
- 13.1kgm@ 4600rpm
- 14.9 KGM at 3000 RPM
- 11.4 kgm at 4,000 rpm
- 12.5@ 3,500(kgm@ rpm)
- 11.8@ 3,200(kgm@ rpm)
- 14.9@ 3,000(kgm@ rpm)
--------------------------------------------------
kgm + rpm 범위
- 22.4 kgm at 1750-2750rpm
- 20.4@ 1400-3400(kgm@ rpm)
- 24@ 1,900-2,750(kgm@ rpm)
- 24 KGM at 1900-2750 RPM
- 14.3@ 1,800-3,000(kgm@ rpm)
- 25.5@ 1,500-3,000(kgm@ rpm)
- 25@ 1,80

In [137]:
# torque 조사 후에도 전체 행과 열 수가 유지되는지 확인합니다.
df_preprocessed.shape

(6926, 13)

In [138]:
# torque 조사 후에도 완전 중복 행이 없는지 확인합니다.
df_preprocessed.duplicated().sum()

np.int64(0)

In [139]:
# kgm only 첫 번째 숫자의 최소·최대 고유값 10개를 생략 없이 확인합니다.
print("가장 작은 고유값 10개:")
print(sorted(torque_first_number[kgm_only_mask].dropna().unique())[:10])
print("가장 큰 고유값 10개:")
print(sorted(
    torque_first_number[kgm_only_mask].dropna().unique(),
    reverse=True,
)[:10])

가장 작은 고유값 10개:
[np.float64(4.8), np.float64(5.7), np.float64(6.0), np.float64(6.1), np.float64(7.8), np.float64(8.5), np.float64(8.6), np.float64(9.2), np.float64(9.8), np.float64(10.2)]
가장 큰 고유값 10개:
[np.float64(190.0), np.float64(145.0), np.float64(130.0), np.float64(115.0), np.float64(110.0), np.float64(53.0), np.float64(51.0), np.float64(46.5), np.float64(42.0), np.float64(36.6)]


In [140]:
# Nm only 첫 번째 숫자의 최소·최대 고유값 10개를 생략 없이 확인합니다.
print("가장 작은 고유값 10개:")
print(sorted(torque_first_number[nm_only_mask].dropna().unique())[:10])
print("가장 큰 고유값 10개:")
print(sorted(
    torque_first_number[nm_only_mask].dropna().unique(),
    reverse=True,
)[:10])

가장 작은 고유값 10개:
[np.float64(48.0), np.float64(51.0), np.float64(57.0), np.float64(59.0), np.float64(60.0), np.float64(62.0), np.float64(69.0), np.float64(71.0), np.float64(72.0), np.float64(72.9)]
가장 큰 고유값 10개:
[np.float64(789.0), np.float64(640.0), np.float64(620.0), np.float64(619.0), np.float64(600.0), np.float64(580.0), np.float64(560.0), np.float64(550.0), np.float64(540.0), np.float64(510.0)]


In [141]:
# kgm only 중 첫 번째 숫자가 50 이상인 조사 대상을 정의합니다.
large_kgm_mask = (
    kgm_only_mask
    & torque_first_number.ge(50)
)

large_kgm_mask.sum()

np.int64(23)

In [142]:
# 큰 kgm 첫 번째 숫자별 행 수를 확인합니다.
torque_first_number[large_kgm_mask].value_counts().sort_index()

torque
51.0      1
53.0      1
110.0     1
115.0     5
130.0     4
145.0    10
190.0     1
Name: count, dtype: int64

In [143]:
# 큰 kgm 그룹의 원본 torque 문자열별 빈도를 모두 확인합니다.
torque_text[large_kgm_mask].value_counts()

torque
145@ 4,100(kgm@ rpm)         10
115@ 2,500(kgm@ rpm)          4
130@ 2500(kgm@ rpm)           4
115@ 2500(kgm@ rpm)           1
110@ 3,000(kgm@ rpm)          1
190@ 21,800(kgm@ rpm)         1
51@ 1,750-3,000(kgm@ rpm)     1
53@ 2,000-2,750(kgm@ rpm)     1
Name: count, dtype: int64

In [144]:
# 큰 kgm 조사 대상 차량을 torque 문자열별 최대 10행 확인합니다.
large_kgm_rows = df_preprocessed.loc[
    large_kgm_mask,
    ["name", "year", "fuel", "engine", "max_power", "torque"],
].copy()
large_kgm_rows["torque_first_number"] = torque_first_number.loc[
    large_kgm_rows.index
]

(
    large_kgm_rows
    .sort_values(["torque_first_number", "torque", "name", "year"])
    .groupby("torque", group_keys=False)
    .head(10)
    [["name", "year", "fuel", "engine", "max_power", "torque"]]
)

,name,year,fuel,engine,max_power,torque
5985,BMW 5 Series 530d,2013,Diesel,2993.0,235.00,"51@ 1,750-3,000(kgm@ rpm)"
6597,BMW X5 3.0d,2008,Diesel,2993.0,235.00,"53@ 2,000-2,750(kgm@ rpm)"
4996,Mahindra Logan Petrol 1.4 GLE,2010,Petrol,1390.0,75.00,"110@ 3,000(kgm@ rpm)"
604,Tata Sumo CX 10 Str BSIV,2011,Diesel,1948.0,68.00,"115@ 2,500(kgm@ rpm)"
4971,Tata Sumo CX 9 Seater,2008,Diesel,1948.0,68.00,"115@ 2,500(kgm@ rpm)"
5242,Tata Sumo EX 10/7 Str BSIII,2008,Diesel,1948.0,68.00,"115@ 2,500(kgm@ rpm)"
2086,Tata Sumo EX 10/7 Str BSIII,2012,Diesel,1948.0,68.00,"115@ 2,500(kgm@ rpm)"
4454,Tata Sumo GX 7 Str BSII,2006,Diesel,1948.0,68.00,115@ 2500(kgm@ rpm)
2880,Ford Ikon 1.6 EXi NXt,2003,Petrol,1597.0,92.00,130@ 2500(kgm@ rpm)
6772,Ford Ikon 1.6 Nxt,2004,Petrol,1597.0,92.00,130@ 2500(kgm@ rpm)


In [145]:
# 조사 대상 name별 다른 torque 기록과 단위 유형을 비교합니다.
for name, target_group in large_kgm_rows.groupby("name", sort=True):
    same_name_mask = df_preprocessed["name"].eq(name) & torque_text.notna()
    target_values = sorted(target_group["torque"].dropna().unique().tolist())
    all_values = torque_text[same_name_mask].drop_duplicates().tolist()
    other_values = [value for value in all_values if value not in target_values]

    print(f"name: {name}")
    print(f"조사 대상 torque: {target_values}")
    print(f"동일 name의 다른 torque: {other_values[:10]}")
    print(f"동일 name에서 Nm 표기 존재: {bool((same_name_mask & nm_only_mask).any())}")
    print(f"동일 name에서 50 미만 kgm 표기 존재: {bool((same_name_mask & kgm_only_mask & torque_first_number.lt(50)).any())}")
    print(f"동일 name에서 큰 kgm 표기 존재: {bool((same_name_mask & large_kgm_mask).any())}")
    print("-" * 70)

name: BMW 5 Series 530d
조사 대상 torque: ['51@ 1,750-3,000(kgm@ rpm)']
동일 name의 다른 torque: []
동일 name에서 Nm 표기 존재: False
동일 name에서 50 미만 kgm 표기 존재: False
동일 name에서 큰 kgm 표기 존재: True
----------------------------------------------------------------------
name: BMW X5 3.0d
조사 대상 torque: ['53@ 2,000-2,750(kgm@ rpm)']
동일 name의 다른 torque: []
동일 name에서 Nm 표기 존재: False
동일 name에서 50 미만 kgm 표기 존재: False
동일 name에서 큰 kgm 표기 존재: True
----------------------------------------------------------------------
name: Ford Ikon 1.6 EXi NXt
조사 대상 torque: ['130@ 2500(kgm@ rpm)']
동일 name의 다른 torque: []
동일 name에서 Nm 표기 존재: False
동일 name에서 50 미만 kgm 표기 존재: False
동일 name에서 큰 kgm 표기 존재: True
----------------------------------------------------------------------
name: Ford Ikon 1.6 Nxt
조사 대상 torque: ['130@ 2500(kgm@ rpm)']
동일 name의 다른 torque: []
동일 name에서 Nm 표기 존재: False
동일 name에서 50 미만 kgm 표기 존재: False
동일 name에서 큰 kgm 표기 존재: True
----------------------------------------------------------------------
name: Ford Ikon 1.

In [146]:
# 조사 대상 name + year별 다른 torque 기록과 Nm 표기를 비교합니다.
for (name, year), target_group in large_kgm_rows.groupby(["name", "year"], sort=True):
    same_name_year_mask = (
        df_preprocessed["name"].eq(name)
        & df_preprocessed["year"].eq(year)
        & torque_text.notna()
    )
    target_values = sorted(target_group["torque"].dropna().unique().tolist())
    all_values = torque_text[same_name_year_mask].drop_duplicates().tolist()
    other_values = [value for value in all_values if value not in target_values]

    print(f"name: {name}")
    print(f"year: {year}")
    print(f"조사 대상 torque: {target_values}")
    print(f"동일 name + year의 다른 torque 존재: {bool(other_values)}")
    print(f"동일 name + year의 다른 torque: {other_values[:10]}")
    print(f"동일 name + year에서 Nm 표기 존재: {bool((same_name_year_mask & nm_only_mask).any())}")
    print("-" * 70)

name: BMW 5 Series 530d
year: 2013
조사 대상 torque: ['51@ 1,750-3,000(kgm@ rpm)']
동일 name + year의 다른 torque 존재: False
동일 name + year의 다른 torque: []
동일 name + year에서 Nm 표기 존재: False
----------------------------------------------------------------------
name: BMW X5 3.0d
year: 2008
조사 대상 torque: ['53@ 2,000-2,750(kgm@ rpm)']
동일 name + year의 다른 torque 존재: False
동일 name + year의 다른 torque: []
동일 name + year에서 Nm 표기 존재: False
----------------------------------------------------------------------
name: Ford Ikon 1.6 EXi NXt
year: 2003
조사 대상 torque: ['130@ 2500(kgm@ rpm)']
동일 name + year의 다른 torque 존재: False
동일 name + year의 다른 torque: []
동일 name + year에서 Nm 표기 존재: False
----------------------------------------------------------------------
name: Ford Ikon 1.6 Nxt
year: 2004
조사 대상 torque: ['130@ 2500(kgm@ rpm)']
동일 name + year의 다른 torque 존재: False
동일 name + year의 다른 torque: []
동일 name + year에서 Nm 표기 존재: False
----------------------------------------------------------------------
name: Ford Ikon 1.

In [147]:
# 큰 kgm 숫자와 동일한 첫 숫자가 Nm only에도 등장하는지 확인합니다.
large_kgm_numbers = sorted(
    torque_first_number[large_kgm_mask].dropna().unique().tolist()
)

for number in large_kgm_numbers:
    kgm_same_number_mask = large_kgm_mask & torque_first_number.eq(number)
    nm_same_number_mask = nm_only_mask & torque_first_number.eq(number)
    nm_examples = df_preprocessed.loc[
        nm_same_number_mask,
        ["name", "year", "torque"],
    ].head(5)

    print(f"첫 숫자: {number}")
    print(f"kgm only 행 수: {int(kgm_same_number_mask.sum())}")
    print(f"Nm only 동일 숫자 행 수: {int(nm_same_number_mask.sum())}")
    print("Nm only 차량 예시:")
    print(nm_examples.to_string(index=True))
    print("-" * 70)

첫 숫자: 51.0
kgm only 행 수: 1
Nm only 동일 숫자 행 수: 18
Nm only 차량 예시:
               name  year               torque
363   Tata Nano STD  2012        51Nm@ 4000rpm
1217   Tata Nano CX  2013  51Nm@ 4000+/-500rpm
1425  Tata Nano XTA  2015        51Nm@ 4000rpm
2265   Tata Nano CX  2013  51Nm@ 4000+/-500rpm
2422   Tata Nano LX  2015  51Nm@ 4000+/-500rpm
----------------------------------------------------------------------
첫 숫자: 53.0
kgm only 행 수: 1
Nm only 동일 숫자 행 수: 0
Nm only 차량 예시:
Empty DataFrame
Columns: [name, year, torque]
Index: []
----------------------------------------------------------------------
첫 숫자: 110.0
kgm only 행 수: 1
Nm only 동일 숫자 행 수: 77
Nm only 차량 예시:
                                         name  year          torque
91        Volkswagen Polo 1.2 MPI Comfortline  2015  110Nm@ 3750rpm
246             Honda Amaze S CVT Petrol BSIV  2019  110Nm@ 4800rpm
353  Volkswagen Ameo 1.2 MPI Comfortline Plus  2019  110Nm@ 3750rpm
378                             Honda Jazz VX  2016  110

In [148]:
# 50 미만과 50 이상 kgm 그룹의 문자열 형식을 비교합니다.
small_kgm_mask = kgm_only_mask & torque_first_number.lt(50)
kgm_parenthetical_unit_mask = torque_lower.str.contains(
    r"\(\s*kgm@\s*rpm\)",
    regex=True,
    na=False,
)
kgm_direct_attached_mask = torque_lower.str.contains(
    r"[0-9](?:\.[0-9]+)?kgm",
    regex=True,
    na=False,
)
kgm_at_word_format_mask = torque_lower.str.contains(
    r"\s+kgm\s+at\s+",
    regex=True,
    na=False,
)

pd.DataFrame({
    "전체 행 수": [small_kgm_mask.sum(), large_kgm_mask.sum()],
    "괄호 포함": [
        (small_kgm_mask & torque_parenthesis_mask).sum(),
        (large_kgm_mask & torque_parenthesis_mask).sum(),
    ],
    "(kgm@ rpm) 포함": [
        (small_kgm_mask & kgm_parenthetical_unit_mask).sum(),
        (large_kgm_mask & kgm_parenthetical_unit_mask).sum(),
    ],
    "숫자 뒤 직접 kgm": [
        (small_kgm_mask & kgm_direct_attached_mask).sum(),
        (large_kgm_mask & kgm_direct_attached_mask).sum(),
    ],
    '" kgm at " 형태': [
        (small_kgm_mask & kgm_at_word_format_mask).sum(),
        (large_kgm_mask & kgm_at_word_format_mask).sum(),
    ],
}, index=["50 미만", "50 이상"])

,전체 행 수,괄호 포함,(kgm@ rpm) 포함,숫자 뒤 직접 kgm,""" kgm at "" 형태"
50 미만,457,352,352,25,80
50 이상,23,23,23,0,0


In [149]:
# 큰 kgm 값 중 괄호가 포함된 고유 문자열과 빈도를 확인합니다.
large_kgm_parenthesis_mask = large_kgm_mask & torque_parenthesis_mask

torque_text[large_kgm_parenthesis_mask].value_counts()

torque
145@ 4,100(kgm@ rpm)         10
115@ 2,500(kgm@ rpm)          4
130@ 2500(kgm@ rpm)           4
115@ 2500(kgm@ rpm)           1
110@ 3,000(kgm@ rpm)          1
190@ 21,800(kgm@ rpm)         1
51@ 1,750-3,000(kgm@ rpm)     1
53@ 2,000-2,750(kgm@ rpm)     1
Name: count, dtype: int64

In [150]:
# 큰 kgm 괄호형 문자열의 차량을 문자열별 최대 10행 확인합니다.
(
    df_preprocessed.loc[
        large_kgm_parenthesis_mask,
        ["name", "year", "fuel", "engine", "max_power", "torque"],
    ]
    .sort_values(["torque", "name", "year"])
    .groupby("torque", group_keys=False)
    .head(10)
)

,name,year,fuel,engine,max_power,torque
4996,Mahindra Logan Petrol 1.4 GLE,2010,Petrol,1390.0,75.00,"110@ 3,000(kgm@ rpm)"
604,Tata Sumo CX 10 Str BSIV,2011,Diesel,1948.0,68.00,"115@ 2,500(kgm@ rpm)"
4971,Tata Sumo CX 9 Seater,2008,Diesel,1948.0,68.00,"115@ 2,500(kgm@ rpm)"
5242,Tata Sumo EX 10/7 Str BSIII,2008,Diesel,1948.0,68.00,"115@ 2,500(kgm@ rpm)"
2086,Tata Sumo EX 10/7 Str BSIII,2012,Diesel,1948.0,68.00,"115@ 2,500(kgm@ rpm)"
4454,Tata Sumo GX 7 Str BSII,2006,Diesel,1948.0,68.00,115@ 2500(kgm@ rpm)
2880,Ford Ikon 1.6 EXi NXt,2003,Petrol,1597.0,92.00,130@ 2500(kgm@ rpm)
6772,Ford Ikon 1.6 Nxt,2004,Petrol,1597.0,92.00,130@ 2500(kgm@ rpm)
2421,Ford Ikon 1.6 Nxt,2009,Petrol,1597.0,92.00,130@ 2500(kgm@ rpm)
6918,Ford Ikon 1.6 Really Sport,2003,Petrol,1597.0,92.00,130@ 2500(kgm@ rpm)


In [151]:
# 큰 kgm 괄호형 차량의 동일 name torque 기록을 비교합니다.
large_parenthesis_rows = df_preprocessed.loc[
    large_kgm_parenthesis_mask,
    ["name", "year", "torque"],
]

for name, target_group in large_parenthesis_rows.groupby("name", sort=True):
    same_name_mask = df_preprocessed["name"].eq(name) & torque_text.notna()
    target_values = sorted(target_group["torque"].dropna().unique().tolist())
    all_values = torque_text[same_name_mask].drop_duplicates().tolist()
    other_values = [value for value in all_values if value not in target_values]

    print(f"name: {name}")
    print(f"큰 kgm 괄호형 torque: {target_values}")
    print(f"동일 name의 다른 torque: {other_values[:10]}")
    print(f"동일 name에서 Nm 표기 존재: {bool((same_name_mask & nm_only_mask).any())}")
    print("-" * 70)

name: BMW 5 Series 530d
큰 kgm 괄호형 torque: ['51@ 1,750-3,000(kgm@ rpm)']
동일 name의 다른 torque: []
동일 name에서 Nm 표기 존재: False
----------------------------------------------------------------------
name: BMW X5 3.0d
큰 kgm 괄호형 torque: ['53@ 2,000-2,750(kgm@ rpm)']
동일 name의 다른 torque: []
동일 name에서 Nm 표기 존재: False
----------------------------------------------------------------------
name: Ford Ikon 1.6 EXi NXt
큰 kgm 괄호형 torque: ['130@ 2500(kgm@ rpm)']
동일 name의 다른 torque: []
동일 name에서 Nm 표기 존재: False
----------------------------------------------------------------------
name: Ford Ikon 1.6 Nxt
큰 kgm 괄호형 torque: ['130@ 2500(kgm@ rpm)']
동일 name의 다른 torque: []
동일 name에서 Nm 표기 존재: False
----------------------------------------------------------------------
name: Ford Ikon 1.6 Really Sport
큰 kgm 괄호형 torque: ['130@ 2500(kgm@ rpm)']
동일 name의 다른 torque: []
동일 name에서 Nm 표기 존재: False
----------------------------------------------------------------------
name: Mahindra Logan Petrol 1.4 GLE
큰 kgm 괄호형 torqu

In [152]:
# 큰 kgm 조사 후에도 전체 행과 열 수가 유지되는지 확인합니다.
df_preprocessed.shape

(6926, 13)

In [153]:
# 큰 kgm 조사 후에도 완전 중복 행이 없는지 확인합니다.
df_preprocessed.duplicated().sum()

np.int64(0)

In [154]:
# torque를 안전 처리 가능 그룹과 보류 그룹으로 나눕니다.
both_mask = both_unit_mask
neither_mask = neither_unit_mask

torque_safe_nm_mask = nm_only_mask
torque_safe_kgm_mask = (
    kgm_only_mask
    & torque_first_number.lt(50)
)
torque_ambiguous_large_kgm_mask = (
    kgm_only_mask
    & torque_first_number.ge(50)
)
torque_both_unit_mask = both_mask
torque_no_unit_mask = neither_mask

pd.Series({
    "Nm only": torque_safe_nm_mask.sum(),
    "kgm only & < 50": torque_safe_kgm_mask.sum(),
    "kgm only & >= 50": torque_ambiguous_large_kgm_mask.sum(),
    "both": torque_both_unit_mask.sum(),
    "no unit": torque_no_unit_mask.sum(),
})

Nm only             6226
kgm only & < 50      457
kgm only & >= 50      23
both                   1
no unit               10
dtype: int64

In [155]:
# 다섯 그룹의 합계, 겹침, 미분류 non-null 행을 확인합니다.
torque_group_membership_count = pd.concat(
    [
        torque_safe_nm_mask,
        torque_safe_kgm_mask,
        torque_ambiguous_large_kgm_mask,
        torque_both_unit_mask,
        torque_no_unit_mask,
    ],
    axis=1,
).sum(axis=1)

pd.Series({
    "다섯 그룹 합계": int(torque_group_membership_count.sum()),
    "non-null torque": int(df_preprocessed["torque"].notna().sum()),
    "두 그룹 이상 겹치는 행": int(torque_group_membership_count.gt(1).sum()),
    "미분류 non-null 행": int(
        (
            df_preprocessed["torque"].notna()
            & torque_group_membership_count.eq(0)
        ).sum()
    ),
})

다섯 그룹 합계           6717
non-null torque    6717
두 그룹 이상 겹치는 행         0
미분류 non-null 행        0
dtype: int64

In [156]:
# 안전하게 해석 가능한 torque만 임시 Nm 후보 Series에 저장합니다.
torque_nm_candidate = pd.Series(
    pd.NA,
    index=df_preprocessed.index,
    dtype="Float64",
)

torque_nm_candidate.loc[torque_safe_nm_mask] = (
    torque_first_number.loc[torque_safe_nm_mask]
)
torque_nm_candidate.loc[torque_safe_kgm_mask] = (
    torque_first_number.loc[torque_safe_kgm_mask]
    * 9.80665
)
torque_nm_candidate.loc[torque_both_unit_mask] = (
    torque_first_number.loc[torque_both_unit_mask]
)

In [157]:
# 안전 처리 가능한 torque 행 수와 비율을 확인합니다.
safe_count = torque_nm_candidate.notna().sum()

pd.Series({
    "안전 처리 가능 행": safe_count,
    "전체 데이터 대비 비율": safe_count / len(df_preprocessed),
    "non-null torque 대비 비율": (
        safe_count / df_preprocessed["torque"].notna().sum()
    ),
})

안전 처리 가능 행               6684.000000
전체 데이터 대비 비율                0.965059
non-null torque 대비 비율       0.995087
dtype: float64

In [158]:
# 아직 해석하지 않는 세 그룹의 행 수와 합계를 확인합니다.
unresolved_torque_counts = pd.Series({
    "원래 torque 결측": df_preprocessed["torque"].isna().sum(),
    "큰 kgm": torque_ambiguous_large_kgm_mask.sum(),
    "단위 없음": torque_no_unit_mask.sum(),
})

pd.concat([
    unresolved_torque_counts,
    pd.Series({"합계": unresolved_torque_counts.sum()}),
])

원래 torque 결측    209
큰 kgm            23
단위 없음            10
합계              242
dtype: int64

In [159]:
# 임시 Nm 후보값의 분포를 확인합니다.
torque_nm_candidate.describe()

count        6684.0
mean     170.870219
std       84.440761
min        47.07192
25%           110.0
50%           160.0
75%       200.05566
max           789.0
dtype: Float64

In [160]:
# 임시 Nm 후보값의 가장 작은 값과 가장 큰 고유값을 확인합니다.
smallest_torque_nm_values = sorted(
    torque_nm_candidate.dropna().unique().tolist()
)[:10]
largest_torque_nm_values = sorted(
    torque_nm_candidate.dropna().unique().tolist(),
    reverse=True,
)[:10]

print("가장 작은 고유값 10개:")
print(smallest_torque_nm_values)
print("가장 큰 고유값 10개:")
print(largest_torque_nm_values)

가장 작은 고유값 10개:
[47.07192, 48.0, 51.0, 55.897905, 57.0, 58.8399, 59.0, 59.820564999999995, 60.0, 62.0]
가장 큰 고유값 10개:
[789.0, 640.0, 620.0, 619.0, 600.0, 580.0, 560.0, 550.0, 540.0, 510.0]


In [161]:
# 임시 Nm 후보값이 가장 작은 10개 값의 차량 예시를 값별 최대 5행 확인합니다.
torque_candidate_examples = df_preprocessed.loc[
    torque_nm_candidate.notna(),
    ["name", "year", "engine", "max_power", "torque"],
].copy()
torque_candidate_examples["torque_nm_candidate"] = torque_nm_candidate.loc[
    torque_candidate_examples.index
]

for value in smallest_torque_nm_values:
    examples = torque_candidate_examples.loc[
        torque_candidate_examples["torque_nm_candidate"].eq(value)
    ].head(5)
    print(f"임시 Nm 후보값: {value}")
    print(examples.to_string(index=True))
    print("-" * 70)

임시 Nm 후보값: 47.07192
              name  year  engine  max_power           torque  torque_nm_candidate
5878  Tata Nano Lx  2011   624.0       35.0  4.8kgm@ 3000rpm             47.07192
----------------------------------------------------------------------


임시 Nm 후보값: 48.0
                   name  year  engine  max_power                    torque  torque_nm_candidate
709        Tata Nano Cx  2011   624.0       35.0             48Nm@ 3000rpm                 48.0
1267  Tata Nano Cx BSIV  2010   624.0       35.0  48@ 3,000+/-500(NM@ rpm)                 48.0
2770       Tata Nano Cx  2009   624.0       35.0             48Nm@ 3000rpm                 48.0
6738  Tata Nano Lx BSIV  2010   624.0       35.0  48@ 3,000+/-500(NM@ rpm)                 48.0
7624       Tata Nano Cx  2011   624.0       35.0             48Nm@ 3000rpm                 48.0
----------------------------------------------------------------------
임시 Nm 후보값: 51.0
               name  year  engine  max_power               torque  torque_nm_candidate
363   Tata Nano STD  2012   624.0      37.50        51Nm@ 4000rpm                 51.0
1217   Tata Nano CX  2013   624.0      37.48  51Nm@ 4000+/-500rpm                 51.0
1425  Tata Nano XTA  2015   624.0      37.48        51Nm@ 40

임시 Nm 후보값: 62.0
                     name  year  engine  max_power         torque  torque_nm_candidate
18         Maruti Alto LX  2002   796.0       46.3  62Nm@ 3000rpm                 62.0
98   Maruti Alto LX BSIII  2008   796.0       46.3  62Nm@ 3000rpm                 62.0
126       Maruti Alto LXi  2008   796.0       46.3  62Nm@ 3000rpm                 62.0
220       Maruti Alto LXi  2011   796.0       46.3  62Nm@ 3000rpm                 62.0
284        Maruti Alto LX  2005   796.0       46.3  62Nm@ 3000rpm                 62.0
----------------------------------------------------------------------


In [162]:
# 임시 Nm 후보값이 가장 큰 10개 값의 차량 예시를 값별 최대 5행 확인합니다.
for value in largest_torque_nm_values:
    examples = torque_candidate_examples.loc[
        torque_candidate_examples["torque_nm_candidate"].eq(value)
    ].head(5)
    print(f"임시 Nm 후보값: {value}")
    print(examples.to_string(index=True))
    print("-" * 70)

임시 Nm 후보값: 789.0
              name  year  engine  max_power          torque  torque_nm_candidate
951   Maruti Zen D  2003  1527.0       58.0  789Nm@ 2250rpm                789.0
4720  Maruti Zen D  2002  1527.0       58.0  789Nm@ 2250rpm                789.0
5143  Maruti Zen D  2006  1527.0       58.0  789Nm@ 2250rpm                789.0
----------------------------------------------------------------------
임시 Nm 후보값: 640.0
                              name  year  engine  max_power          torque  torque_nm_candidate
170  Volvo XC90 T8 Excellence BSIV  2017  1969.0      400.0  640Nm@ 1740rpm                640.0
----------------------------------------------------------------------
임시 Nm 후보값: 620.0
                                  name  year  engine  max_power               torque  torque_nm_candidate
136    Mercedes-Benz S-Class S 350 CDI  2017  2987.0     254.79  620Nm@ 1600-2400rpm                620.0
1071  BMW 6 Series GT 630d Luxury Line  2018  2993.0     261.40  620Nm@ 2000-

In [163]:
# kgm < 50 그룹을 Nm로 변환한 값의 분포를 확인합니다.
torque_nm_candidate[torque_safe_kgm_mask].describe()

count         457.0
mean     163.069354
std        60.11656
min        47.07192
25%      122.583125
50%        156.9064
75%       200.05566
max      456.009225
dtype: Float64

In [164]:
# kgm < 50 변환 결과의 가장 작은 값과 가장 큰 고유값을 확인합니다.
safe_kgm_nm_values = torque_nm_candidate[torque_safe_kgm_mask].dropna()

print("가장 작은 변환값 10개:")
print(sorted(safe_kgm_nm_values.unique().tolist())[:10])
print("가장 큰 변환값 10개:")
print(sorted(safe_kgm_nm_values.unique().tolist(), reverse=True)[:10])

가장 작은 변환값 10개:
[47.07192, 55.897905, 58.8399, 59.820564999999995, 76.49186999999999, 83.35652499999999, 84.33718999999999, 90.22117999999999, 96.10517, 100.02782999999998]
가장 큰 변환값 10개:
[456.00922499999996, 411.8793, 358.92339, 350.097405, 330.484105, 323.61945, 314.79346499999997, 277.528195, 250.069575, 245.16625]


In [165]:
# 원래 Nm only와 kgm < 50 변환 그룹의 분포를 비교합니다.
nm_only_candidate_values = torque_nm_candidate[torque_safe_nm_mask].dropna()
converted_kgm_candidate_values = torque_nm_candidate[
    torque_safe_kgm_mask
].dropna()

pd.DataFrame({
    "count": [
        nm_only_candidate_values.count(),
        converted_kgm_candidate_values.count(),
    ],
    "min": [
        nm_only_candidate_values.min(),
        converted_kgm_candidate_values.min(),
    ],
    "median": [
        nm_only_candidate_values.median(),
        converted_kgm_candidate_values.median(),
    ],
    "mean": [
        nm_only_candidate_values.mean(),
        converted_kgm_candidate_values.mean(),
    ],
    "max": [
        nm_only_candidate_values.max(),
        converted_kgm_candidate_values.max(),
    ],
}, index=["Nm only", "kgm < 50 → Nm 변환"])

,count,min,median,mean,max
Nm only,6226,48.00000,160.0000,171.409227,789.000000
kgm < 50 → Nm 변환,457,47.07192,156.9064,163.069354,456.009225


In [166]:
# torque 후보값 조사 후에도 전체 행과 열 수가 유지되는지 확인합니다.
df_preprocessed.shape

(6926, 13)

In [167]:
# torque 후보값 조사 후에도 완전 중복 행이 없는지 확인합니다.
df_preprocessed.duplicated().sum()

np.int64(0)

In [168]:
# torque Nm 후보값이 큰 순서대로 상위 30행을 확인합니다.
torque_candidate_rows = df_preprocessed.loc[
    torque_nm_candidate.notna(),
    ["name", "year", "fuel", "engine", "max_power", "torque"],
].copy()
torque_candidate_rows["torque_nm_candidate"] = torque_nm_candidate.loc[
    torque_nm_candidate.notna()
]

top_30_torque_candidate_rows = torque_candidate_rows.sort_values(
    "torque_nm_candidate",
    ascending=False,
).head(30)

top_30_torque_candidate_rows

,name,year,fuel,engine,max_power,torque,torque_nm_candidate
4720,Maruti Zen D,2002,Diesel,1527.0,58.00,789Nm@ 2250rpm,789.0
5143,Maruti Zen D,2006,Diesel,1527.0,58.00,789Nm@ 2250rpm,789.0
951,Maruti Zen D,2003,Diesel,1527.0,58.00,789Nm@ 2250rpm,789.0
170,Volvo XC90 T8 Excellence BSIV,2017,Petrol,1969.0,400.00,640Nm@ 1740rpm,640.0
4766,BMW 6 Series GT 630d Luxury Line,2018,Diesel,2993.0,261.40,620Nm@ 2000-2500rpm,620.0
1071,BMW 6 Series GT 630d Luxury Line,2018,Diesel,2993.0,261.40,620Nm@ 2000-2500rpm,620.0
4753,BMW 6 Series GT 630d Luxury Line,2018,Diesel,2993.0,261.40,620Nm@ 2000-2500rpm,620.0
6258,BMW 6 Series GT 630d Luxury Line,2018,Diesel,2993.0,261.40,620Nm@ 2000-2500rpm,620.0
136,Mercedes-Benz S-Class S 350 CDI,2017,Diesel,2987.0,254.79,620Nm@ 1600-2400rpm,620.0
2938,BMW X7 xDrive 30d DPE,2020,Diesel,2993.0,265.00,620Nm@ 1500-2500rpm,620.0


In [169]:
# 가장 큰 torque 후보 고유값 20개의 빈도를 확인합니다.
largest_20_torque_values = sorted(
    torque_nm_candidate.dropna().unique().tolist(),
    reverse=True,
)[:20]

largest_20_torque_value_counts = (
    torque_nm_candidate[
        torque_nm_candidate.isin(largest_20_torque_values)
    ]
    .value_counts()
    .reindex(largest_20_torque_values)
)

largest_20_torque_value_counts

789.000000     3
640.000000     1
620.000000     6
619.000000     3
600.000000     3
580.000000     2
560.000000     2
550.000000     6
540.000000     1
510.000000     1
500.000000     8
490.000000     1
480.000000     1
470.000000     4
456.009225     1
450.000000    10
436.400000     1
436.390000     2
430.000000     3
424.000000     2
Name: count, dtype: Int64

In [170]:
# torque Nm 후보값이 789인 행을 모두 확인합니다.
torque_789_mask = torque_nm_candidate.eq(789)
torque_789_rows = df_preprocessed.loc[
    torque_789_mask,
    ["name", "year", "fuel", "engine", "max_power", "torque"],
].copy()

print(f"789 Nm 행 수: {int(torque_789_mask.sum())}")
torque_789_rows

789 Nm 행 수: 3


,name,year,fuel,engine,max_power,torque
951,Maruti Zen D,2003,Diesel,1527.0,58.0,789Nm@ 2250rpm
4720,Maruti Zen D,2002,Diesel,1527.0,58.0,789Nm@ 2250rpm
5143,Maruti Zen D,2006,Diesel,1527.0,58.0,789Nm@ 2250rpm


In [171]:
# 789 Nm 행과 같은 name, name + year, engine + max_power의 다른 torque 기록을 확인합니다.
target_789_torque_values = set(torque_789_rows["torque"].dropna().unique())

print("[동일 name 기준]")
for name in torque_789_rows["name"].drop_duplicates():
    same_name_torque = df_preprocessed.loc[
        df_preprocessed["name"].eq(name),
        "torque",
    ].dropna()
    other_torque = same_name_torque[
        ~same_name_torque.isin(target_789_torque_values)
    ]
    print(f"name: {name}")
    print(f"전체 고유 torque: {same_name_torque.unique().tolist()}")
    print(f"다른 torque 존재 여부: {not other_torque.empty}")
    print(f"다른 torque와 빈도: {other_torque.value_counts().to_dict()}")

print("\n[동일 name + year 기준]")
for name, year in torque_789_rows[["name", "year"]].drop_duplicates().itertuples(index=False):
    same_name_year_torque = df_preprocessed.loc[
        df_preprocessed["name"].eq(name)
        & df_preprocessed["year"].eq(year),
        "torque",
    ].dropna()
    other_torque = same_name_year_torque[
        ~same_name_year_torque.isin(target_789_torque_values)
    ]
    print(f"name: {name}, year: {year}")
    print(f"전체 고유 torque: {same_name_year_torque.unique().tolist()}")
    print(f"다른 torque 존재 여부: {not other_torque.empty}")
    print(f"다른 torque와 빈도: {other_torque.value_counts().to_dict()}")

print("\n[동일 engine + max_power 기준]")
for engine, max_power in torque_789_rows[["engine", "max_power"]].drop_duplicates().itertuples(index=False):
    same_spec_torque = df_preprocessed.loc[
        df_preprocessed["engine"].eq(engine)
        & df_preprocessed["max_power"].eq(max_power),
        "torque",
    ].dropna()
    other_torque = same_spec_torque[
        ~same_spec_torque.isin(target_789_torque_values)
    ]
    print(f"engine: {engine}, max_power: {max_power}")
    print(f"전체 고유 torque: {same_spec_torque.unique().tolist()}")
    print(f"다른 torque 존재 여부: {not other_torque.empty}")
    print(f"다른 torque와 빈도: {other_torque.value_counts().to_dict()}")

[동일 name 기준]
name: Maruti Zen D
전체 고유 torque: ['789Nm@ 2250rpm']
다른 torque 존재 여부: False
다른 torque와 빈도: {}

[동일 name + year 기준]
name: Maruti Zen D, year: 2003
전체 고유 torque: ['789Nm@ 2250rpm']
다른 torque 존재 여부: False
다른 torque와 빈도: {}
name: Maruti Zen D, year: 2002
전체 고유 torque: ['789Nm@ 2250rpm']
다른 torque 존재 여부: False
다른 torque와 빈도: {}
name: Maruti Zen D, year: 2006
전체 고유 torque: ['789Nm@ 2250rpm']
다른 torque 존재 여부: False
다른 torque와 빈도: {}

[동일 engine + max_power 기준]
engine: 1527.0, max_power: 58.0
전체 고유 torque: ['789Nm@ 2250rpm']
다른 torque 존재 여부: False
다른 torque와 빈도: {}


In [172]:
# torque 후보값과 max_power의 진단용 비율을 계산합니다.
valid_torque_power_ratio_mask = (
    torque_nm_candidate.notna()
    & df_preprocessed["max_power"].gt(0)
)

torque_power_ratio = pd.Series(
    pd.NA,
    index=df_preprocessed.index,
    dtype="Float64",
)
torque_power_ratio.loc[valid_torque_power_ratio_mask] = (
    torque_nm_candidate.loc[valid_torque_power_ratio_mask]
    / df_preprocessed.loc[valid_torque_power_ratio_mask, "max_power"]
)

torque_power_ratio.describe()

count       6684.0
mean      1.912281
std       0.575417
min       0.899083
25%       1.381418
50%       1.988072
75%       2.357143
max      13.603448
dtype: Float64

In [173]:
# torque / max_power 진단 비율이 큰 순서대로 상위 30행을 확인합니다.
torque_ratio_rows = df_preprocessed.loc[
    torque_power_ratio.notna(),
    ["name", "year", "fuel", "engine", "max_power", "torque"],
].copy()
torque_ratio_rows["torque_nm_candidate"] = torque_nm_candidate.loc[
    torque_power_ratio.notna()
]
torque_ratio_rows["torque_power_ratio"] = torque_power_ratio.loc[
    torque_power_ratio.notna()
]

top_30_torque_power_ratio_rows = torque_ratio_rows.sort_values(
    "torque_power_ratio",
    ascending=False,
).head(30)

top_30_torque_power_ratio_rows

,name,year,fuel,engine,max_power,torque,torque_nm_candidate,torque_power_ratio
5143,Maruti Zen D,2006,Diesel,1527.0,58.00,789Nm@ 2250rpm,789.0,13.603448
951,Maruti Zen D,2003,Diesel,1527.0,58.00,789Nm@ 2250rpm,789.0,13.603448
4720,Maruti Zen D,2002,Diesel,1527.0,58.00,789Nm@ 2250rpm,789.0,13.603448
2137,Land Rover Freelander 2 TD4 HSE,2013,Diesel,2179.0,115.00,400 Nm /2000 rpm,400.0,3.478261
4654,Tata Sumo Gold GX BSIII,2012,Diesel,2956.0,68.40,223Nm@ 1600-2200rpm,223.0,3.260234
5587,Tata Sumo Gold CX BSIII,2017,Diesel,2956.0,69.01,223Nm@ 1600-2200rpm,223.0,3.231416
277,Tata Sumo Gold EX BSIII,2015,Diesel,2956.0,69.01,223Nm@ 1600-2200rpm,223.0,3.231416
838,Tata Sumo Gold EX BSIII,2017,Diesel,2956.0,69.01,223Nm@ 1600-2200rpm,223.0,3.231416
1515,Mahindra Bolero 2011-2019 Plus AC,2012,Diesel,2523.0,62.10,195Nm@ 1440-2200rpm,195.0,3.140097
1497,Mahindra Bolero 2011-2019 SLX 2WD BSIII,2013,Diesel,2523.0,62.10,195Nm@ 1400-2200rpm,195.0,3.140097


In [174]:
# 789 Nm 행의 진단 비율과 전체 분포 내 백분위 위치를 확인합니다.
torque_power_ratio_percentile = torque_power_ratio.rank(
    method="average",
    pct=True,
)

torque_789_ratio_rows = df_preprocessed.loc[
    torque_789_mask,
    ["name", "year", "max_power", "torque"],
].copy()
torque_789_ratio_rows["torque_nm_candidate"] = torque_nm_candidate.loc[
    torque_789_mask
]
torque_789_ratio_rows["torque_power_ratio"] = torque_power_ratio.loc[
    torque_789_mask
]
torque_789_ratio_rows["ratio_percentile"] = torque_power_ratio_percentile.loc[
    torque_789_mask
]

torque_789_ratio_rows

,name,year,max_power,torque,torque_nm_candidate,torque_power_ratio,ratio_percentile
951,Maruti Zen D,2003,58.0,789Nm@ 2250rpm,789.0,13.603448,0.99985
4720,Maruti Zen D,2002,58.0,789Nm@ 2250rpm,789.0,13.603448,0.99985
5143,Maruti Zen D,2006,58.0,789Nm@ 2250rpm,789.0,13.603448,0.99985


In [175]:
# 가장 큰 torque 후보값 20개별 행 수와 서로 다른 name 개수를 확인합니다.
largest_20_torque_summary = pd.DataFrame(
    [
        {
            "torque_nm_candidate": value,
            "행 수": int(torque_nm_candidate.eq(value).sum()),
            "서로 다른 name 수": int(
                df_preprocessed.loc[
                    torque_nm_candidate.eq(value),
                    "name",
                ].nunique()
            ),
        }
        for value in largest_20_torque_values
    ]
)

largest_20_torque_summary

,torque_nm_candidate,행 수,서로 다른 name 수
0,789.000000,3,1
1,640.000000,1,1
2,620.000000,6,3
3,619.000000,3,2
4,600.000000,3,2
5,580.000000,2,2
6,560.000000,2,1
7,550.000000,6,3
8,540.000000,1,1
9,510.000000,1,1


In [176]:
# torque Nm 후보값이 작은 순서대로 하위 20행을 확인합니다.
bottom_20_torque_candidate_rows = torque_candidate_rows.sort_values(
    "torque_nm_candidate",
    ascending=True,
).head(20)

bottom_20_torque_candidate_rows

,name,year,fuel,engine,max_power,torque,torque_nm_candidate
5878,Tata Nano Lx,2011,Petrol,624.0,35.00,4.8kgm@ 3000rpm,47.07192
709,Tata Nano Cx,2011,Petrol,624.0,35.00,48Nm@ 3000rpm,48.0
7624,Tata Nano Cx,2011,Petrol,624.0,35.00,48Nm@ 3000rpm,48.0
2770,Tata Nano Cx,2009,Petrol,624.0,35.00,48Nm@ 3000rpm,48.0
1267,Tata Nano Cx BSIV,2010,Petrol,624.0,35.00,"48@ 3,000+/-500(NM@ rpm)",48.0
6738,Tata Nano Lx BSIV,2010,Petrol,624.0,35.00,"48@ 3,000+/-500(NM@ rpm)",48.0
8089,Tata Nano Cx,2011,Petrol,624.0,35.00,48Nm@ 3000rpm,48.0
7616,Tata Nano Lx BSIV,2012,Petrol,624.0,37.48,51Nm@ 4000+/-500rpm,51.0
6817,Tata Nano Lx BSIV,2012,Petrol,624.0,37.48,51Nm@ 4000+/-500rpm,51.0
7833,Tata Nano LX SE,2012,Petrol,624.0,37.50,51Nm@ 4000+/-500rpm,51.0


In [177]:
# 추가 조사 후에도 전체 행과 열 수가 유지되는지 확인합니다.
df_preprocessed.shape

(6926, 13)

In [178]:
# 추가 조사 후에도 완전 중복 행이 없는지 확인합니다.
df_preprocessed.duplicated().sum()

np.int64(0)

In [179]:
# 아직 단위를 확정하지 못한 torque 행을 정의합니다.
torque_unresolved_mask = (
    torque_ambiguous_large_kgm_mask
    | torque_no_unit_mask
)

int(torque_unresolved_mask.sum())

33

In [180]:
# 미확정 33행과 각 행의 조사 그룹을 확인합니다.
torque_unresolved_index = torque_unresolved_mask.index[
    torque_unresolved_mask
]

torque_unresolved_group = pd.Series(
    pd.NA,
    index=df_preprocessed.index,
    dtype="string",
)
torque_unresolved_group.loc[
    torque_ambiguous_large_kgm_mask.index[
        torque_ambiguous_large_kgm_mask
    ]
] = "large_kgm"
torque_unresolved_group.loc[
    torque_no_unit_mask.index[torque_no_unit_mask]
] = "no_unit"

torque_unresolved_rows = df_preprocessed.loc[
    torque_unresolved_index,
    ["name", "year", "fuel", "engine", "max_power", "torque"],
].copy()
torque_unresolved_rows.insert(
    0,
    "group",
    torque_unresolved_group.loc[torque_unresolved_index],
)

torque_unresolved_rows

,group,name,year,fuel,engine,max_power,torque
140,no_unit,Skoda Superb LK 1.8 TSI AT,2018,Petrol,1798.0,177.46,250@ 1250-5000rpm
604,large_kgm,Tata Sumo CX 10 Str BSIV,2011,Diesel,1948.0,68.00,"115@ 2,500(kgm@ rpm)"
1211,large_kgm,Maruti SX4 Vxi BSIII,2007,Petrol,1586.0,104.68,"145@ 4,100(kgm@ rpm)"
1676,no_unit,Mercedes-Benz M-Class ML 350 4Matic,2011,Diesel,2987.0,165.00,510@ 1600-2400
2000,no_unit,Honda Jazz Select Edition Active,2011,Petrol,1198.0,90.00,110(11.2)@ 4800
2086,large_kgm,Tata Sumo EX 10/7 Str BSIII,2012,Diesel,1948.0,68.00,"115@ 2,500(kgm@ rpm)"
2289,large_kgm,Maruti SX4 Zxi BSIII,2010,Petrol,1586.0,104.68,"145@ 4,100(kgm@ rpm)"
2314,large_kgm,Maruti SX4 Zxi with Leather BSIII,2008,Petrol,1586.0,104.68,"145@ 4,100(kgm@ rpm)"
2421,large_kgm,Ford Ikon 1.6 Nxt,2009,Petrol,1597.0,92.00,130@ 2500(kgm@ rpm)
2880,large_kgm,Ford Ikon 1.6 EXi NXt,2003,Petrol,1597.0,92.00,130@ 2500(kgm@ rpm)


In [181]:
# 외부 검증 시 사용할 고유 조합과 원본 행 수를 계산합니다.
validation_key_columns = [
    "group",
    "name",
    "year",
    "engine",
    "max_power",
    "torque",
]

torque_validation_targets = (
    torque_unresolved_rows
    .reset_index(names="source_index")
    .groupby(validation_key_columns, dropna=False)
    .size()
    .rename("row_count")
    .reset_index()
)

group_order = {"large_kgm": 0, "no_unit": 1}
torque_validation_targets["_group_order"] = (
    torque_validation_targets["group"].map(group_order)
)
torque_validation_targets = (
    torque_validation_targets
    .sort_values(["_group_order", "name", "year"])
    .drop(columns="_group_order")
    .reset_index(drop=True)
)

torque_validation_targets

,group,name,year,engine,max_power,torque,row_count
0,large_kgm,BMW 5 Series 530d,2013,2993.0,235.00,"51@ 1,750-3,000(kgm@ rpm)",1
1,large_kgm,BMW X5 3.0d,2008,2993.0,235.00,"53@ 2,000-2,750(kgm@ rpm)",1
2,large_kgm,Ford Ikon 1.6 EXi NXt,2003,1597.0,92.00,130@ 2500(kgm@ rpm),1
3,large_kgm,Ford Ikon 1.6 Nxt,2004,1597.0,92.00,130@ 2500(kgm@ rpm),1
4,large_kgm,Ford Ikon 1.6 Nxt,2009,1597.0,92.00,130@ 2500(kgm@ rpm),1
5,large_kgm,Ford Ikon 1.6 Really Sport,2003,1597.0,92.00,130@ 2500(kgm@ rpm),1
6,large_kgm,Mahindra Logan Petrol 1.4 GLE,2010,1390.0,75.00,"110@ 3,000(kgm@ rpm)",1
7,large_kgm,Maruti SX4 Vxi BSIII,2007,1586.0,104.68,"145@ 4,100(kgm@ rpm)",2
8,large_kgm,Maruti SX4 Vxi BSIII,2008,1586.0,104.68,"145@ 4,100(kgm@ rpm)",1
9,large_kgm,Maruti SX4 Vxi BSIII,2009,1586.0,104.68,"145@ 4,100(kgm@ rpm)",2


In [182]:
# 미확정 행과 고유 검증 대상의 개수를 확인합니다.
validation_target_counts = pd.Series({
    "원본 미확정 행 수": int(torque_unresolved_mask.sum()),
    "고유 name 수": int(torque_unresolved_rows["name"].nunique()),
    "고유 name + torque 조합 수": int(
        torque_unresolved_rows[["name", "torque"]]
        .drop_duplicates()
        .shape[0]
    ),
    "고유 전체 검증 조합 수": int(len(torque_validation_targets)),
    "large_kgm 고유 검증 대상 수": int(
        torque_validation_targets["group"].eq("large_kgm").sum()
    ),
    "no_unit 고유 검증 대상 수": int(
        torque_validation_targets["group"].eq("no_unit").sum()
    ),
})

validation_target_counts

원본 미확정 행 수               33
고유 name 수                21
고유 name + torque 조합 수    21
고유 전체 검증 조합 수            28
large_kgm 고유 검증 대상 수     19
no_unit 고유 검증 대상 수        9
dtype: int64

In [183]:
# 미확정 33행의 torque 원본 문자열별 반복 구조를 요약합니다.
torque_string_summary = (
    torque_unresolved_rows
    .groupby("torque", dropna=False)
    .agg(
        row_count=("name", "size"),
        unique_name_count=("name", "nunique"),
        min_year=("year", "min"),
        max_year=("year", "max"),
    )
    .sort_values(["row_count", "torque"], ascending=[False, True])
    .reset_index()
)

torque_string_summary

,torque,row_count,unique_name_count,min_year,max_year
0,"145@ 4,100(kgm@ rpm)",10,3,2007,2010
1,210 / 1900,7,4,2003,2010
2,"115@ 2,500(kgm@ rpm)",4,3,2008,2012
3,130@ 2500(kgm@ rpm),4,3,2003,2009
4,110(11.2)@ 4800,1,1,2011,2011
5,"110@ 3,000(kgm@ rpm)",1,1,2010,2010
6,115@ 2500(kgm@ rpm),1,1,2006,2006
7,"190@ 21,800(kgm@ rpm)",1,1,2005,2005
8,250@ 1250-5000rpm,1,1,2018,2018
9,510@ 1600-2400,1,1,2011,2011


In [184]:
# 동일 engine + max_power 조합에 다른 torque 문자열이 있는지 확인합니다.
engine_power_comparison_records = []

for target in torque_validation_targets.itertuples(index=False):
    if pd.isna(target.engine):
        engine_match = df_preprocessed["engine"].isna()
    else:
        engine_match = df_preprocessed["engine"].eq(target.engine)

    if pd.isna(target.max_power):
        max_power_match = df_preprocessed["max_power"].isna()
    else:
        max_power_match = df_preprocessed["max_power"].eq(
            target.max_power
        )

    same_spec_torque = df_preprocessed.loc[
        engine_match
        & max_power_match
        & df_preprocessed["torque"].notna(),
        "torque",
    ]
    other_torque_values = sorted(
        same_spec_torque[
            ~same_spec_torque.eq(target.torque)
        ].unique().tolist()
    )

    engine_power_comparison_records.append({
        "name": target.name,
        "year": target.year,
        "engine": target.engine,
        "max_power": target.max_power,
        "unresolved_torque": target.torque,
        "other_torque_values": other_torque_values,
        "has_other_torque": bool(other_torque_values),
    })

engine_power_torque_comparison = pd.DataFrame(
    engine_power_comparison_records
)

engine_power_torque_comparison

,name,year,engine,max_power,unresolved_torque,other_torque_values,has_other_torque
0,BMW 5 Series 530d,2013,2993.0,235.00,"51@ 1,750-3,000(kgm@ rpm)","[53@ 2,000-2,750(kgm@ rpm)]",True
1,BMW X5 3.0d,2008,2993.0,235.00,"53@ 2,000-2,750(kgm@ rpm)","[51@ 1,750-3,000(kgm@ rpm)]",True
2,Ford Ikon 1.6 EXi NXt,2003,1597.0,92.00,130@ 2500(kgm@ rpm),[],False
3,Ford Ikon 1.6 Nxt,2004,1597.0,92.00,130@ 2500(kgm@ rpm),[],False
4,Ford Ikon 1.6 Nxt,2009,1597.0,92.00,130@ 2500(kgm@ rpm),[],False
5,Ford Ikon 1.6 Really Sport,2003,1597.0,92.00,130@ 2500(kgm@ rpm),[],False
6,Mahindra Logan Petrol 1.4 GLE,2010,1390.0,75.00,"110@ 3,000(kgm@ rpm)","[110Nm@ 3000rpm, 11@ 3,000(kgm@ rpm)]",True
7,Maruti SX4 Vxi BSIII,2007,1586.0,104.68,"145@ 4,100(kgm@ rpm)",[],False
8,Maruti SX4 Vxi BSIII,2008,1586.0,104.68,"145@ 4,100(kgm@ rpm)",[],False
9,Maruti SX4 Vxi BSIII,2009,1586.0,104.68,"145@ 4,100(kgm@ rpm)",[],False


In [185]:
# 외부 검색에 사용할 최종 고유 검증 대상 표를 확인합니다.
validation_fuel = (
    torque_unresolved_rows
    .reset_index(names="source_index")
    .groupby(validation_key_columns, dropna=False)["fuel"]
    .agg(lambda values: ", ".join(sorted(values.dropna().unique())))
    .rename("fuel")
    .reset_index()
)

final_torque_validation_targets = torque_validation_targets.merge(
    validation_fuel,
    on=validation_key_columns,
    how="left",
)
final_torque_validation_targets["_group_order"] = (
    final_torque_validation_targets["group"].map(group_order)
)
final_torque_validation_targets = (
    final_torque_validation_targets
    .sort_values(["_group_order", "name", "year"])
    .drop(columns="_group_order")
    [[
        "group",
        "name",
        "year",
        "fuel",
        "engine",
        "max_power",
        "torque",
        "row_count",
    ]]
    .reset_index(drop=True)
)

final_torque_validation_targets

,group,name,year,fuel,engine,max_power,torque,row_count
0,large_kgm,BMW 5 Series 530d,2013,Diesel,2993.0,235.00,"51@ 1,750-3,000(kgm@ rpm)",1
1,large_kgm,BMW X5 3.0d,2008,Diesel,2993.0,235.00,"53@ 2,000-2,750(kgm@ rpm)",1
2,large_kgm,Ford Ikon 1.6 EXi NXt,2003,Petrol,1597.0,92.00,130@ 2500(kgm@ rpm),1
3,large_kgm,Ford Ikon 1.6 Nxt,2004,Petrol,1597.0,92.00,130@ 2500(kgm@ rpm),1
4,large_kgm,Ford Ikon 1.6 Nxt,2009,Petrol,1597.0,92.00,130@ 2500(kgm@ rpm),1
5,large_kgm,Ford Ikon 1.6 Really Sport,2003,Petrol,1597.0,92.00,130@ 2500(kgm@ rpm),1
6,large_kgm,Mahindra Logan Petrol 1.4 GLE,2010,Petrol,1390.0,75.00,"110@ 3,000(kgm@ rpm)",1
7,large_kgm,Maruti SX4 Vxi BSIII,2007,Petrol,1586.0,104.68,"145@ 4,100(kgm@ rpm)",2
8,large_kgm,Maruti SX4 Vxi BSIII,2008,Petrol,1586.0,104.68,"145@ 4,100(kgm@ rpm)",1
9,large_kgm,Maruti SX4 Vxi BSIII,2009,Petrol,1586.0,104.68,"145@ 4,100(kgm@ rpm)",2


In [186]:
# 검증 대상 정리 후에도 데이터 크기와 완전 중복 수가 유지되는지 확인합니다.
pd.Series({
    "행 수": df_preprocessed.shape[0],
    "열 수": df_preprocessed.shape[1],
    "완전 중복 행 수": int(df_preprocessed.duplicated().sum()),
})

행 수          6926
열 수            13
완전 중복 행 수       0
dtype: int64

In [187]:
# 기존 Nm 후보값을 복사해 최종 규칙 검증용 Series를 만듭니다.
torque_nm_final_candidate = torque_nm_candidate.copy()

In [188]:
# 789Nm 원본 문자열 3행을 최종 후보에서 결측 처리합니다.
invalid_789_mask = torque_text.eq("789Nm@ 2250rpm")
invalid_789_index = invalid_789_mask.index[invalid_789_mask]

torque_nm_final_candidate.loc[invalid_789_index] = pd.NA

int(invalid_789_mask.sum())

3

In [189]:
# BMW 두 문자열은 첫 번째 숫자를 실제 kgm로 해석해 Nm로 변환합니다.
bmw_actual_kgm_strings = [
    "51@ 1,750-3,000(kgm@ rpm)",
    "53@ 2,000-2,750(kgm@ rpm)",
]
bmw_actual_kgm_mask = torque_text.isin(bmw_actual_kgm_strings)
bmw_actual_kgm_index = bmw_actual_kgm_mask.index[
    bmw_actual_kgm_mask
]

torque_nm_final_candidate.loc[bmw_actual_kgm_index] = (
    torque_first_number.loc[bmw_actual_kgm_index] * 9.80665
)

pd.DataFrame({
    "row_count": torque_text[
        bmw_actual_kgm_mask
    ].value_counts().reindex(bmw_actual_kgm_strings),
    "torque_nm_final_candidate": [
        torque_nm_final_candidate.loc[
            torque_text[torque_text.eq(value)].index[0]
        ]
        for value in bmw_actual_kgm_strings
    ],
})

,row_count,torque_nm_final_candidate
torque,,
"51@ 1,750-3,000(kgm@ rpm)",1,500.13915
"53@ 2,000-2,750(kgm@ rpm)",1,519.75245


In [190]:
# BMW 두 문자열을 제외한 큰 kgm 21행은 첫 번째 숫자를 Nm로 사용합니다.
remaining_large_kgm_mask = (
    torque_ambiguous_large_kgm_mask
    & ~bmw_actual_kgm_mask
)
remaining_large_kgm_index = remaining_large_kgm_mask.index[
    remaining_large_kgm_mask
]

torque_nm_final_candidate.loc[remaining_large_kgm_index] = (
    torque_first_number.loc[remaining_large_kgm_index]
)

int(remaining_large_kgm_mask.sum())

21

In [191]:
# 단위가 없는 10행은 첫 번째 숫자를 Nm 후보값으로 사용합니다.
torque_no_unit_index = torque_no_unit_mask.index[
    torque_no_unit_mask
]
torque_nm_final_candidate.loc[torque_no_unit_index] = (
    torque_first_number.loc[torque_no_unit_index]
)

int(torque_no_unit_mask.sum())

10

In [192]:
# 최종 후보값 수와 결측값 구성을 확인합니다.
expected_final_missing_mask = (
    df_preprocessed["torque"].isna()
    | df_preprocessed["torque"].eq("789Nm@ 2250rpm")
)

final_candidate_counts = pd.Series({
    "유효 후보값": int(torque_nm_final_candidate.notna().sum()),
    "결측 후보값": int(torque_nm_final_candidate.isna().sum()),
    "원래 torque 결측": int(df_preprocessed["torque"].isna().sum()),
    "789Nm 결측": int(
        torque_nm_final_candidate.loc[invalid_789_index].isna().sum()
    ),
    "큰 kgm 또는 no_unit 미해석 잔여": int(
        torque_nm_final_candidate.loc[
            torque_unresolved_index
        ].isna().sum()
    ),
    "예상 구성 외 결측": int(
        (
            torque_nm_final_candidate.isna()
            & ~expected_final_missing_mask
        ).sum()
    ),
})

final_candidate_counts

유효 후보값                     6714
결측 후보값                      212
원래 torque 결측                209
789Nm 결측                      3
큰 kgm 또는 no_unit 미해석 잔여       0
예상 구성 외 결측                    0
dtype: int64

In [193]:
# 최종 Nm 후보값의 분포를 확인합니다.
torque_nm_final_candidate.describe()

count        6714.0
mean     170.678349
std        83.58833
min        47.07192
25%           110.0
50%           160.0
75%       200.05566
max           640.0
dtype: Float64

In [194]:
# 최종 후보값의 가장 큰 값과 가장 작은 고유값을 확인합니다.
largest_10_final_torque_values = sorted(
    torque_nm_final_candidate.dropna().unique().tolist(),
    reverse=True,
)[:10]
smallest_10_final_torque_values = sorted(
    torque_nm_final_candidate.dropna().unique().tolist()
)[:10]

print("가장 큰 고유값 10개:")
print(largest_10_final_torque_values)
print("가장 작은 고유값 10개:")
print(smallest_10_final_torque_values)

가장 큰 고유값 10개:
[640.0, 620.0, 619.0, 600.0, 580.0, 560.0, 550.0, 540.0, 519.75245, 510.0]
가장 작은 고유값 10개:
[47.07192, 48.0, 51.0, 55.897905, 57.0, 58.8399, 59.0, 59.820564999999995, 60.0, 62.0]


In [195]:
# 최종 후보값 상·하위 고유값에 해당하는 대표 차량을 값별 최대 5행 확인합니다.
final_extreme_vehicle_examples = []

for range_name, values in [
    ("largest", largest_10_final_torque_values),
    ("smallest", smallest_10_final_torque_values),
]:
    for value in values:
        value_mask = torque_nm_final_candidate.eq(value)
        examples = df_preprocessed.loc[
            value_mask,
            ["name", "year", "engine", "max_power", "torque"],
        ].head(5).copy()
        examples.insert(0, "range", range_name)
        examples["torque_nm_final_candidate"] = value
        final_extreme_vehicle_examples.append(examples)

final_extreme_vehicle_examples = pd.concat(
    final_extreme_vehicle_examples,
    ignore_index=False,
)

final_extreme_vehicle_examples

,range,name,year,engine,max_power,torque,torque_nm_final_candidate
170,largest,Volvo XC90 T8 Excellence BSIV,2017,1969.0,400.00,640Nm@ 1740rpm,640.000000
136,largest,Mercedes-Benz S-Class S 350 CDI,2017,2987.0,254.79,620Nm@ 1600-2400rpm,620.000000
1071,largest,BMW 6 Series GT 630d Luxury Line,2018,2993.0,261.40,620Nm@ 2000-2500rpm,620.000000
2938,largest,BMW X7 xDrive 30d DPE,2020,2993.0,265.00,620Nm@ 1500-2500rpm,620.000000
4753,largest,BMW 6 Series GT 630d Luxury Line,2018,2993.0,261.40,620Nm@ 2000-2500rpm,620.000000
4766,largest,BMW 6 Series GT 630d Luxury Line,2018,2993.0,261.40,620Nm@ 2000-2500rpm,620.000000
2705,largest,Mercedes-Benz M-Class ML 350 CDI,2013,2987.0,254.80,619Nm@ 1600-2400rpm,619.000000
7506,largest,Mercedes-Benz GL-Class 350 CDI Blue Efficiency,2014,2987.0,254.80,619Nm@ 1600-2400rpm,619.000000
7808,largest,Mercedes-Benz M-Class ML 350 CDI,2014,2987.0,254.80,619Nm@ 1600-2400rpm,619.000000
2826,largest,Jaguar XF 3.0 Litre S Premium Luxury,2014,2993.0,270.90,600Nm@ 2000rpm,600.000000


In [196]:
# 새로 해석한 미확정 33행의 최종 후보값을 확인합니다.
unresolved_final_check = df_preprocessed.loc[
    torque_unresolved_index,
    ["name", "torque"],
].copy()
unresolved_final_check["torque_nm_final_candidate"] = (
    torque_nm_final_candidate.loc[torque_unresolved_index]
)

non_bmw_unresolved_index = torque_unresolved_index.difference(
    bmw_actual_kgm_index
)
non_bmw_matches_first_number = (
    torque_nm_final_candidate.loc[non_bmw_unresolved_index]
    .sub(torque_first_number.loc[non_bmw_unresolved_index])
    .abs()
    .lt(1e-12)
    .sum()
)

print(pd.Series({
    "미확정 33행 중 결측": int(
        torque_nm_final_candidate.loc[
            torque_unresolved_index
        ].isna().sum()
    ),
    "BMW 실제 kgm 변환 행": int(len(bmw_actual_kgm_index)),
    "나머지 첫 숫자와 동일한 Nm 행": int(
        non_bmw_matches_first_number
    ),
}))

unresolved_final_check

미확정 33행 중 결측           0
BMW 실제 kgm 변환 행        2
나머지 첫 숫자와 동일한 Nm 행    31
dtype: int64


,name,torque,torque_nm_final_candidate
140,Skoda Superb LK 1.8 TSI AT,250@ 1250-5000rpm,250.0
604,Tata Sumo CX 10 Str BSIV,"115@ 2,500(kgm@ rpm)",115.0
1211,Maruti SX4 Vxi BSIII,"145@ 4,100(kgm@ rpm)",145.0
1676,Mercedes-Benz M-Class ML 350 4Matic,510@ 1600-2400,510.0
2000,Honda Jazz Select Edition Active,110(11.2)@ 4800,110.0
2086,Tata Sumo EX 10/7 Str BSIII,"115@ 2,500(kgm@ rpm)",115.0
2289,Maruti SX4 Zxi BSIII,"145@ 4,100(kgm@ rpm)",145.0
2314,Maruti SX4 Zxi with Leather BSIII,"145@ 4,100(kgm@ rpm)",145.0
2421,Ford Ikon 1.6 Nxt,130@ 2500(kgm@ rpm),130.0
2880,Ford Ikon 1.6 EXi NXt,130@ 2500(kgm@ rpm),130.0


In [197]:
# 789Nm 3행이 최종 후보에서 모두 결측인지 확인합니다.
invalid_789_final_check = df_preprocessed.loc[
    invalid_789_index,
    ["name", "year", "torque"],
].copy()
invalid_789_final_check["torque_nm_final_candidate"] = (
    torque_nm_final_candidate.loc[invalid_789_index]
)

print(
    "789Nm 최종 후보 결측 행:",
    int(invalid_789_final_check["torque_nm_final_candidate"].isna().sum()),
)
invalid_789_final_check

789Nm 최종 후보 결측 행: 3


,name,year,torque,torque_nm_final_candidate
951,Maruti Zen D,2003,789Nm@ 2250rpm,<NA>
4720,Maruti Zen D,2002,789Nm@ 2250rpm,<NA>
5143,Maruti Zen D,2006,789Nm@ 2250rpm,<NA>


In [198]:
# 최종 후보 검증 후에도 데이터 크기와 완전 중복 수가 유지되는지 확인합니다.
pd.Series({
    "행 수": df_preprocessed.shape[0],
    "열 수": df_preprocessed.shape[1],
    "완전 중복 행 수": int(df_preprocessed.duplicated().sum()),
})

행 수          6926
열 수            13
완전 중복 행 수       0
dtype: int64